# Part 4 - Azure DevOps Work Items

## Goal

From the planning document:

- *"After approval, a Planning Agent creates user stories, acceptance criteria, and proposed ownership."*
- *"A second MCP connection will expose Azure DevOps operations for reading workloads and creating linked work items."*
- *"After final approval, the system ... creates linked Azure DevOps items, recommending assignees from role,
  specialization, workload, and deadline data rather than assigning them automatically."*

This notebook builds:

1. The **Planning Agent**, which turns the approved specification into user stories with Given/When/Then
   acceptance criteria and FUNC/TECH tasks.
2. An **Azure DevOps client** that uses the REST API when credentials are configured and a local dry-run board
   otherwise.
3. The **owner recommendation**, based on role, specialization, open workload and the deadline.
4. An **Azure DevOps MCP server** whose create tool refuses to run without a recorded human approval.
5. **Support emails to work items**, the day-to-day flow: the customer's support **Feature** is found from
   the sender's email domain (and remembered for next time), the **Support Planning Agent** splits the email
   into small tasks with an **Activity, estimate and due date**, the email's **attachments** are uploaded to the
   User Story, and the **acknowledgement email** with the ticket number is drafted and **sent** through the
   email server of notebook 03. When the email asks for a change rather than a fix, a **Change Request** in the
   company form is attached to the story and to the email. The documents are filed on **SharePoint**
   (`<root>/<Customer>/<Story ID - Description>/<Document type>/`) and the folder is linked on the story.
   Every write needs a human approval.

## Setup

Same environment as the course notebooks: a local **Ollama** server running **`gemma4:e4b`** for the agents and **`nomic-embed-text`** for embeddings. The cells work on Google Colab (Ollama is installed automatically) and locally (install Ollama from ollama.com first).

This notebook only needs the chat model.

**Chat model:** set `LLM_PROVIDER=openai` in `.env` to run the agents on **GPT Luna** (`OPENAI_MODEL`, default `gpt-6-luna`, key in `OPEN_AI_KEY`); Ollama is then not needed for this notebook. `LLM_PROVIDER=ollama` (the default) keeps the local `gemma4:e4b`.

In [ ]:
%pip install -q -U langchain langchain-core langchain-ollama langchain-openai python-dotenv msal langchain-text-splitters langgraph langchain-mcp-adapters mcp python-docx requests numpy langsmith

In [ ]:
import os
import platform
import shutil
import subprocess
import time

import requests
from dotenv import load_dotenv

load_dotenv()  # reads .env next to the notebook (OPEN_AI_KEY, LLM_PROVIDER, ADO_*, ...) when it exists

# Chat model: "openai" runs the agents on GPT Luna (key in OPEN_AI_KEY), "ollama" on the local model.
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()
if LLM_PROVIDER not in ("openai", "ollama"):
    raise ValueError(f"LLM_PROVIDER must be 'openai' or 'ollama', not {LLM_PROVIDER!r}")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e4b")
LLM_MODEL = OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
OLLAMA_URL = "http://localhost:11434"
NEEDS_OLLAMA = LLM_PROVIDER == "ollama"  # this notebook only needs the chat model


def ollama_is_running() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).ok
    except requests.RequestException:
        return False


# 1. Install Ollama (Colab / Linux) and start the server in the background
if NEEDS_OLLAMA:
    if shutil.which("ollama") is None:
        if platform.system() == "Linux":
            print("Installing Ollama...")
            !sudo apt-get install -y -qq zstd
            !curl -fsSL https://ollama.com/install.sh | sh
        else:
            print("Install Ollama from https://ollama.com/download, then re-run this cell.")

    if not ollama_is_running():
        print("Starting Ollama server...")
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)  # Wait for server to wake up

    print("Ollama running:", ollama_is_running())
print(f"Chat model: {LLM_PROVIDER} / {LLM_MODEL}")

In [ ]:
# 2. Pull the chat model and the embedding model (only when the agents run on Ollama)
if NEEDS_OLLAMA:
    !ollama pull {LLM_MODEL}
    !ollama pull {EMBED_MODEL}
    !ollama list

### LangSmith tracing

Every agent, LLM call, MCP tool call and LangGraph node is traced in **LangSmith**, so you can open each step
and see its **prompt, output, duration and tokens**. Paste your API key when asked (or store it as the Colab
secret `LANGSMITH_API_KEY`); press Enter to run without tracing.

In [ ]:
import getpass
import os

import langsmith.utils

# LangSmith records every agent, LLM call (full prompt and answer), MCP tool call and LangGraph node,
# with its duration and token usage. Free API key: https://smith.langchain.com -> Settings -> API Keys.
# Note: traces contain the prompts, so customer emails are sent to LangSmith. Use test data or your own workspace.
USE_LANGSMITH = True
LANGSMITH_PROJECT = "fo-requirements-to-delivery"


def read_langsmith_key() -> str:
    if os.getenv("LANGSMITH_API_KEY"):
        return os.environ["LANGSMITH_API_KEY"]
    try:
        from google.colab import userdata  # Colab: add a secret named LANGSMITH_API_KEY

        return userdata.get("LANGSMITH_API_KEY")
    except Exception:
        pass
    try:
        return getpass.getpass("LangSmith API key (press Enter to run without tracing): ").strip()
    except Exception:  # no interactive input available
        return ""


api_key = read_langsmith_key() if USE_LANGSMITH else ""
if api_key:
    os.environ.update(LANGSMITH_API_KEY=api_key, LANGSMITH_TRACING="true", LANGSMITH_PROJECT=LANGSMITH_PROJECT)
    print(f"LangSmith tracing ON -> project '{LANGSMITH_PROJECT}' on https://smith.langchain.com")
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")
langsmith.utils.get_env_var.cache_clear()  # LangSmith caches environment variables

In [ ]:
import getpass
import os


def make_llm():
    """The chat model selected by LLM_PROVIDER: GPT Luna on OpenAI, or the local Ollama model."""
    if LLM_PROVIDER == "openai":
        from langchain_openai import ChatOpenAI

        if not (os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY")):
            os.environ["OPEN_AI_KEY"] = getpass.getpass("OpenAI API key: ").strip()
        # GPT Luna only accepts its default temperature, so none is set.
        return ChatOpenAI(model=LLM_MODEL, api_key=os.getenv("OPEN_AI_KEY") or os.getenv("OPENAI_API_KEY"))

    from langchain_ollama import ChatOllama

    return ChatOllama(model=LLM_MODEL, temperature=0)


llm = make_llm()

In [ ]:
import json

from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import traceable  # names agent steps in LangSmith (does nothing when tracing is off)


def ask_structured(schema, system_prompt: str, user_prompt: str, retries: int = 2):
    """Call the LLM and force the answer into a Pydantic schema. On failure, retry with the error message."""
    # The run name shows up in LangSmith, e.g. "DeliveryPlan" or "GapAnalysis".
    structured_llm = llm.with_structured_output(schema, method="json_schema").with_config(run_name=schema.__name__)
    last_error = None

    for attempt in range(retries + 1):
        try:
            result = structured_llm.invoke(
                [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
            )
            if result is None:
                raise ValueError("The model returned an empty answer.")
            return result
        except Exception as error:
            last_error = error
            print(f"[retry {attempt + 1}] invalid structured output: {error}")
            user_prompt += (
                f"\n\nYour previous answer was invalid ({error}). "
                "Return JSON that matches the schema exactly."
            )

    raise last_error


def ask_text(system_prompt: str, user_prompt: str) -> str:
    return llm.invoke(
        [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    ).text  # plain string for both providers (OpenAI returns content blocks)


def show(title: str, obj) -> None:
    """Pretty-print a heading and a Pydantic object, dict, list or string."""
    print(f"\n========== {title} ==========\n")
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    print(obj if isinstance(obj, str) else json.dumps(obj, indent=2, ensure_ascii=False))
from pathlib import Path

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Requirement(BaseModel):
    req_id: str = Field(description="Stable ID such as REQ-001")
    statement: str = Field(description="One clear, testable requirement sentence")
    source_text: str = Field(description="The customer's words this requirement comes from")
    actor: str = Field(description="Who performs or benefits from the requirement")
    business_rules: list[str] = Field(description="Rules, thresholds or conditions that are actually stated")
    exceptions: list[str] = Field(description="Exceptions or edge cases that are actually stated")
    expected_outcome: str = Field(description="What should happen when the requirement is met")


class RequirementExtraction(BaseModel):
    summary: str = Field(description="Two-sentence summary of the request")
    actors: list[str]
    in_scope: list[str]
    out_of_scope: list[str]
    security_constraints: list[str] = Field(description="Security, approval or segregation-of-duties constraints")
    requirements: list[Requirement]


class Classification(BaseModel):
    module: Literal["Finance", "Supply Chain", "Finance and Supply Chain", "Out of scope"]
    sub_areas: list[str] = Field(description="F&O areas such as Accounts payable, Procurement, Inventory")
    work_type: Literal["Functional", "Technical", "Mixed"]
    design_documents: Literal["FDD", "TDD", "FDD and TDD"]
    risk_level: Literal["Low", "Medium", "High"]
    rationale: str


class Gap(BaseModel):
    gap_id: str = Field(description="Stable ID such as GAP-01")
    req_id: str = Field(description="The requirement this gap belongs to")
    missing_information: str
    why_it_matters: str
    clarification_question: str = Field(description="One clear question the customer can answer")
    assumption_if_unanswered: str


class GapAnalysis(BaseModel):
    gaps: list[Gap]
class AcceptanceCriterion(BaseModel):
    ac_id: str = Field(description="ID such as AC-01")
    req_ids: list[str]
    given: str
    when: str
    then: str


class TaskItem(BaseModel):
    task_id: str = Field(description="ID such as T-01")
    kind: Literal["Functional", "Technical"]
    title: str = Field(description="Starts with FUNC: or TECH:")
    description: str
    req_ids: list[str]
    estimate_hours: float


class UserStory(BaseModel):
    story_id: str = Field(description="ID such as US-01")
    title: str = Field(description="Format: [Module] short goal")
    as_a: str
    i_want: str
    so_that: str
    req_ids: list[str]
    acceptance_criteria: list[AcceptanceCriterion]
    tasks: list[TaskItem]


class DeliveryPlan(BaseModel):
    user_stories: list[UserStory]

## The Approved Specification

The Planning Agent starts only after the customer's answers were confirmed and the specification was approved
(notebooks 01-03). The approved requirements, the classification, the retrieved evidence and the **immutable
facts** are included here so the notebook can run on its own.

In [ ]:
REQUEST_ID = "CR-2026-001"

CUSTOMER_REQUEST = """
Subject: Vendor invoices approval + blocking vendors

Hi team,

Going forward we need vendor invoices above a certain amount to be approved by the
finance manager before they can be posted. Right now anyone in AP can post them and
the auditors flagged it.

Also, vendors that don't have bank details should be blocked so we don't pay them by
mistake. Some of the old vendors are like this too.

Can we have this for all our companies? We would like it live before the quarter close.

Thanks,
Rana Haddad
AP Team Lead, Cedar Retail Group
"""
REQUEST_ID = "CR-2026-001"
DEADLINE = "2026-12-15"

CLASSIFICATION = {
    "module": "Finance",
    "sub_areas": ["Accounts payable"],
    "work_type": "Mixed",
    "design_documents": "FDD and TDD",
    "risk_level": "Medium",
    "rationale": "Approval workflow configuration (functional) plus a vendor payment-hold batch job and validation (technical).",
}

APPROVED_REQUIREMENTS = [
    {"req_id": "REQ-001", "actor": "Finance Manager",
     "statement": "Vendor invoices with a total including tax greater than 10,000 USD (accounting currency) must be approved by the Finance Manager of the legal entity before they can be posted.",
     "business_rules": ["Threshold 10,000 USD incl. tax, accounting currency", "Legal entities USMF and DEMF"],
     "exceptions": [], "expected_outcome": "Invoices above the threshold cannot be posted until approved."},
    {"req_id": "REQ-002", "actor": "AP Clerk",
     "statement": "Vendor invoices of 10,000 USD or less can be posted by AP clerks without approval.",
     "business_rules": ["Threshold 10,000 USD incl. tax"], "exceptions": [],
     "expected_outcome": "Current posting process is unchanged for small invoices."},
    {"req_id": "REQ-003", "actor": "CFO",
     "statement": "If the Finance Manager does not act within 2 business days, the approval escalates to the CFO.",
     "business_rules": ["Escalation after 2 business days"], "exceptions": [],
     "expected_outcome": "Pending approvals do not block posting indefinitely."},
    {"req_id": "REQ-004", "actor": "AP Team",
     "statement": "Existing and new vendors without at least one active bank account are placed on payment hold, which blocks payment generation but still allows invoice registration.",
     "business_rules": ["On hold = Payment", "Applies to existing and new vendors"], "exceptions": [],
     "expected_outcome": "No payment can be generated for vendors without bank details."},
    {"req_id": "REQ-005", "actor": "Auditor",
     "statement": "The approval history (approver, decision, date and time) of each vendor invoice must be visible to auditors.",
     "business_rules": [], "exceptions": [], "expected_outcome": "Auditors can prove who approved each invoice and when."},
    {"req_id": "REQ-006", "actor": "AP Clerk",
     "statement": "AP clerks must not be able to approve vendor invoices they entered themselves.",
     "business_rules": ["Segregation of duties"], "exceptions": [],
     "expected_outcome": "Self-approval is technically prevented."},
]

LOCKED_FACTS = [
    "Request ID: CR-2026-001",
    "Legal entities in scope: USMF and DEMF only",
    "Approval threshold: vendor invoice total including tax greater than 10,000 USD in accounting currency",
    "Approver: Finance Manager of the legal entity; escalate to the CFO after 2 business days",
    "Blocked vendor means On hold = Payment (invoices can still be registered)",
    "Security: AP clerks must not approve invoices they entered",
    "Go-live deadline: 2026-12-15",
    "Cited evidence: MS-WF-01, MS-AP-01, MS-SEC-01, DES-2025-014, DES-2025-031",
]

EVIDENCE = [
    {"doc_id": "MS-WF-01", "title": "Workflow and approvals", "source_type": "microsoft_guidance", "module": "Finance",
     "snippet": "Approval steps can have conditions, such as the invoice total being greater than an amount... time limits and escalation paths... workflow history records who approved and when."},
    {"doc_id": "MS-AP-01", "title": "Vendor invoices and holds", "source_type": "microsoft_guidance", "module": "Finance",
     "snippet": "On hold = Payment blocks payment generation while invoices can still be registered."},
    {"doc_id": "DES-2025-014", "title": "Approved design: purchase invoice approval by amount", "source_type": "approved_design", "module": "Finance",
     "snippet": "Workflow condition on invoice total (accounting currency) > threshold; SoD rule; threshold in a CUS parameter."},
    {"doc_id": "DES-2025-031", "title": "Approved design: automatic hold for vendors with incomplete master data", "source_type": "approved_design", "module": "Finance",
     "snippet": "Recurring SysOperation batch sets On hold = Payment; CoC validation on vendor insert/update; CUS log table."},
]
# Retrieved from the knowledge base (STD-ADO-01)
WORK_ITEM_STANDARD = (
    "Each approved requirement is delivered through at least one User Story. Story titles use the format "
    "'[Module] short goal', and the description uses 'As a <role>, I want <goal> so that <benefit>'. Acceptance "
    "criteria are written as Given/When/Then and reference requirement IDs. Implementation work is split into child "
    "Tasks: functional tasks are prefixed 'FUNC:' and technical tasks 'TECH:'. Tasks carry an estimate in hours. "
    "Assignees are proposed and then confirmed by a person; tools must not assign work automatically."
)

In [ ]:
def handoff_package(state: dict) -> str:
    """The structured package passed at every handoff (Coordination section of the planning document)."""
    package = {
        "request_id": state.get("request_id"),
        "source_text": state.get("source_text", ""),
        "requirements": state.get("requirements", []),
        "classification": state.get("classification"),
        "confirmed_answers": state.get("confirmed_answers", {}),
        "assumptions": state.get("assumptions", []),
        "retrieved_evidence": [
            {**e, "snippet": e["snippet"][:400]} for e in state.get("evidence", [])
        ],
        "approval_status": state.get("approval_status", {}),
        "immutable_facts": state.get("locked_facts", []),
    }
    return json.dumps(package, indent=2, ensure_ascii=False)
state = {
    "request_id": REQUEST_ID,
    "source_text": CUSTOMER_REQUEST,
    "requirements": APPROVED_REQUIREMENTS,
    "classification": CLASSIFICATION,
    "evidence": EVIDENCE,
    "approval_status": {"specification": "approved"},
    "locked_facts": LOCKED_FACTS,
}
print(handoff_package(state)[:1500], "...")

In [ ]:
PLANNING_SYSTEM = """
You are the Planning Agent of a Dynamics 365 F&O delivery team.
Create Azure DevOps user stories from the approved specification.

Rules:
- Every requirement ID must be covered by at least one user story and at least one acceptance criterion.
- Story titles use the format "[Module] short goal". Acceptance criteria use Given/When/Then, IDs AC-01, AC-02, ...
- Split the work into tasks, IDs T-01, T-02, ...: functional tasks start with "FUNC:" (configuration,
  FDD, training) and technical tasks start with "TECH:" (X++ extensions, security objects, batch jobs,
  TDD, unit tests). Give each task an estimate in hours.
- The immutable facts are final: use their exact values and never change them.
"""

FDD_SYSTEM = """
You are the Functional Design Agent of a Dynamics 365 F&O delivery team.
Write the Functional Design Document (FDD) for the approved specification.

- Follow the company design document standard: business context, as-is and to-be process,
  configuration and parameters, roles and security, exceptions, reports and audit, assumptions.
- Section IDs FDD-1, FDD-2, ... Each section lists the requirement IDs it covers, and every requirement
  must appear in at least one section.
- Use only the immutable facts and confirmed answers. Cite evidence IDs in the text, e.g. [MS-WF-01].
"""

TDD_SYSTEM = """
You are the Technical Design Agent of a Dynamics 365 F&O delivery team.
Write the Technical Design Document (TDD) for the approved specification.

- Follow the company design and development standards: solution overview, object list (table and form
  extensions, Chain of Command classes, SysOperation batch classes, security privileges and duties),
  logic, data model changes, error handling and logging, performance, deployment.
- Extensions only (no overlayering), CUS prefix, configurable values in a parameter table.
- Section IDs TDD-1, TDD-2, ... Each section lists the requirement IDs it covers, and every requirement
  must appear in at least one section.
- Use only the immutable facts. Cite evidence IDs in the text, e.g. [DES-2025-031].
"""

TEST_SYSTEM = """
You are the Test Agent of a Dynamics 365 F&O delivery team.
Create test cases for the approved specification, IDs TC-01, TC-02, ...

- Types: Functional, Technical, Integration and Regression.
- Every requirement ID needs at least one test case.
- Threshold rules need boundary tests at, just below and just above the limit.
- Security rules need a negative test proving an unauthorized user is blocked.
- Add regression tests for standard processes that must keep working.
- Use the exact values from the immutable facts.
"""

VALIDATION_SYSTEM = """
You are the Validation Agent of a Dynamics 365 F&O delivery team.
Compare the artefacts with the immutable facts and with each other.

Report only real problems:
- a wrong value (amount, currency, legal entity, role, number of days, hold type, deadline),
- a contradiction between the design, the acceptance criteria and the tests,
- behaviour that is not in the approved requirements.
For each issue name the responsible agent: planning, functional_design, technical_design or tests.
If everything is consistent, return an empty list of issues.
"""

REVISION_INSTRUCTIONS = """
Revise your previous output. It failed validation.

Validation feedback:
{feedback}

Lessons from earlier attempts (reflexion memory):
{reflections}

Return the complete corrected artefact. Keep everything that was already correct.
"""

## Planning Agent

The agent receives the structured handoff package and the company work-item standard. A quick coverage check
follows: if a requirement has no acceptance criterion, the plan goes back to the agent once with that feedback.
(The complete validation and reflexion loop is in notebook 06.)

In [ ]:
def print_plan(plan: dict, owners: dict | None = None) -> None:
    for story in plan["user_stories"]:
        print(f"\n{story['story_id']}  {story['title']}   covers {', '.join(story['req_ids'])}")
        print(f"   As a {story['as_a']}, I want {story['i_want']} so that {story['so_that']}.")
        for ac in story["acceptance_criteria"]:
            print(f"   {ac['ac_id']} ({', '.join(ac['req_ids'])}): Given {ac['given']}, when {ac['when']}, then {ac['then']}")
        for task in story["tasks"]:
            owner = f"  -> recommended: {owners[task['task_id']]['owner']}" if owners and task["task_id"] in owners else ""
            print(f"   - {task['task_id']} {task['title']} [{task['estimate_hours']}h]{owner}")
@traceable(name="Planning Agent")
def plan_delivery(state: dict, feedback: str = "") -> dict:
    user_prompt = f"APPROVED PACKAGE:\n{handoff_package(state)}\n\nWORK ITEM STANDARD:\n{WORK_ITEM_STANDARD}"
    if feedback:
        user_prompt += f"\n\nFix this problem in your plan: {feedback}"
    return ask_structured(DeliveryPlan, PLANNING_SYSTEM, user_prompt).model_dump()


def uncovered_requirements(plan: dict) -> list:
    covered = {r for s in plan["user_stories"] for ac in s["acceptance_criteria"] for r in ac["req_ids"]}
    return [r["req_id"] for r in APPROVED_REQUIREMENTS if r["req_id"] not in covered]


plan = plan_delivery(state)
missing = uncovered_requirements(plan)
if missing:
    print("Requirements without acceptance criteria:", missing, "-> sending back to the Planning Agent")
    plan = plan_delivery(state, feedback=f"These requirements have no acceptance criterion: {', '.join(missing)}")

print("Uncovered requirements:", uncovered_requirements(plan) or "none")
print_plan(plan)

## Team Roster and Current Workload

The owner recommendation needs the team's roles and specializations, and their open work. The roster is a
small JSON file. The open work is read from Azure DevOps; in dry-run mode it comes from a local board seeded
with the work each person already has.

The dry-run board also contains three customer projects, each with a support Feature, for the support flow at
the end of the notebook.

In [ ]:
TEAM = [
    {"name": "Maya Khoury", "role": "Functional Consultant", "specializations": ["Finance"], "capacity_hours_per_week": 30},
    {"name": "Karim Nassar", "role": "Functional Consultant", "specializations": ["Supply Chain", "Finance"], "capacity_hours_per_week": 30},
    {"name": "Jad Mansour", "role": "Developer", "specializations": ["Finance"], "capacity_hours_per_week": 35},
    {"name": "Lara Aoun", "role": "Developer", "specializations": ["Supply Chain", "Finance"], "capacity_hours_per_week": 35},
    {"name": "Nour Fares", "role": "QA Engineer", "specializations": ["Finance", "Supply Chain"], "capacity_hours_per_week": 30},
]
Path("team.json").write_text(json.dumps(TEAM, indent=2), encoding="utf-8")

# Seed the dry-run board with work the team already has in progress, and with customer projects that each
# have a support Feature (used by the support flow at the end).
existing_work = [("Maya Khoury", 120), ("Karim Nassar", 40), ("Jad Mansour", 150), ("Lara Aoun", 60), ("Nour Fares", 20)]
support_features = [(501, "Cedar Retail Group - F&O Support 2026"), (502, "Northwind Traders - F&O Support"),
                    (503, "Fabrikam Foods - F&O Support and Maintenance")]
Path("ado_dry_run_board.json").write_text(json.dumps({
    "next_id": 1000,
    "projects": [title.split(" - ")[0] for _, title in support_features],
    "work_items": [
        {"id": 900 + i, "type": "Task", "parent_id": None,
         "fields": {"System.Title": f"Existing work for {name}", "System.State": "Active",
                    "System.AssignedTo": name, "Microsoft.VSTS.Scheduling.RemainingWork": hours}}
        for i, (name, hours) in enumerate(existing_work)
    ] + [
        {"id": feature_id, "type": "Feature", "parent_id": None,
         "fields": {"System.Title": title, "System.TeamProject": title.split(" - ")[0], "System.State": "Active"}}
        for feature_id, title in support_features
    ],
}, indent=2), encoding="utf-8")

if not (os.getenv("ADO_ORG_URL") and os.getenv("ADO_PAT")):
    Path("customer_features.csv").unlink(missing_ok=True)  # dry-run: start without remembered customers
print("Team and dry-run board ready")

## Azure DevOps Client

`ado_client.py` uses the Azure DevOps REST API (version 7.1):

- create work items with a JSON Patch document (`application/json-patch+json`) and link each child to its
  parent with `System.LinkTypes.Hierarchy-Reverse`,
- read open work with a WIQL query and the work-items batch API,
- list projects and search support Features across the organization, match an email domain to a customer
  project, and remember the chosen Feature per domain (`customer_features.csv`),
- upload attachments and link them to a work item (`AttachedFile`).

Without `ADO_ORG_URL` and `ADO_PAT`, it writes to `ado_dry_run_board.json` instead.
Recommended owners are written into the task description. `System.AssignedTo` is never set.

In [ ]:
%%writefile ado_client.py
"""Azure DevOps Boards client.

Live mode (REST API 7.1) is used when ADO_ORG_URL and ADO_PAT are set. ADO_PROJECT is the default project for
the delivery plan; support work items go to the project of the customer's Feature.
Otherwise every call goes to a local JSON board (dry-run), so the notebook runs without credentials.
"""
import base64
import csv
import html
import json
import os
import re
from difflib import SequenceMatcher
from pathlib import Path
from urllib.parse import quote

import requests

API_VERSION = "7.1"
CLOSED_STATES = {"Closed", "Done", "Removed", "Resolved"}
DRY_RUN_BOARD = Path("ado_dry_run_board.json")
CUSTOMER_FEATURES = Path("customer_features.csv")  # remembered email domain -> support Feature
PRIORITY = {"High": 1, "Medium": 2, "Low": 3}


def email_domain(address: str) -> str:
    """'rana@cedarretail.example' -> 'cedarretail' (the part that names the company)."""
    return address.split("@")[-1].split(".")[0].lower() if "@" in address else address.lower()


def _normalise(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())


def name_similarity(domain: str, name: str) -> float:
    domain, name = _normalise(domain), _normalise(name)
    if not domain or not name:
        return 0.0
    if domain in name or name in domain:  # "cedarretail" in "Cedar Retail Group"
        return 1.0
    return SequenceMatcher(None, domain, name).ratio()


class AzureDevOpsClient:
    def __init__(self):
        self.org_url = os.getenv("ADO_ORG_URL", "").rstrip("/")  # e.g. https://dev.azure.com/my-org
        self.project = os.getenv("ADO_PROJECT", "")
        self.story_type = os.getenv("ADO_STORY_TYPE", "User Story")  # "Product Backlog Item" for Scrum
        self.due_date_field = os.getenv("ADO_DUE_DATE_FIELD", "Custom.DueDate1")  # empty = do not set
        pat = os.getenv("ADO_PAT", "")
        self.live = bool(self.org_url and pat)
        if self.live:
            token = base64.b64encode(f":{pat}".encode()).decode()
            self.headers = {"Authorization": f"Basic {token}"}

    @property
    def mode(self) -> str:
        return "live" if self.live else "dry-run"

    # ---------------- dry-run board ----------------
    def _load_board(self) -> dict:
        if DRY_RUN_BOARD.exists():
            return json.loads(DRY_RUN_BOARD.read_text(encoding="utf-8"))
        return {"next_id": 1000, "projects": [], "work_items": []}

    def _save_board(self, board: dict) -> None:
        DRY_RUN_BOARD.write_text(json.dumps(board, indent=2), encoding="utf-8")

    def _project(self, project: str | None) -> str:
        project = project or self.project
        if not project:
            raise ValueError("No Azure DevOps project: set ADO_PROJECT or pass a project.")
        return project

    def _api(self, path: str, project: str | None = None) -> str:
        return f"{self.org_url}/{quote(self._project(project))}/_apis/wit/{path}"

    def _org_api(self, path: str) -> str:
        return f"{self.org_url}/_apis/{path}"

    def _request(self, method: str, url: str, **kwargs) -> requests.Response:
        response = requests.request(method, url, headers={**self.headers, **kwargs.pop("headers", {})},
                                    timeout=60, **kwargs)
        if not response.ok:
            raise requests.HTTPError(f"Azure DevOps {method} {url.split('?')[0]} -> {response.status_code}: "
                                     f"{response.text[:500]}", response=response)
        return response

    # ---------------- operations ----------------
    def create_work_item(self, work_item_type: str, fields: dict, parent_id: int | None = None,
                         project: str | None = None) -> dict:
        if not self.live:
            board = self._load_board()
            board["next_id"] += 1
            item = {"id": board["next_id"], "type": work_item_type, "parent_id": parent_id,
                    "fields": {"System.TeamProject": project or self.project or "dry-run", **fields}, "relations": []}
            board["work_items"].append(item)
            self._save_board(board)
            return {"id": item["id"], "type": work_item_type, "url": f"dry-run://workitems/{item['id']}"}

        patch = [{"op": "add", "path": f"/fields/{name}", "value": value} for name, value in fields.items()]
        if parent_id:
            patch.append({"op": "add", "path": "/relations/-", "value": {
                "rel": "System.LinkTypes.Hierarchy-Reverse",  # child -> parent link
                "url": self._org_api(f"wit/workItems/{parent_id}"),
            }})
        response = self._request(
            "POST", self._api(f"workitems/${quote(work_item_type)}?api-version={API_VERSION}", project),
            json=patch, headers={"Content-Type": "application/json-patch+json"},
        )
        data = response.json()
        return {"id": data["id"], "type": work_item_type, "url": data["_links"]["html"]["href"]}

    def get_work_item(self, work_item_id: int) -> dict:
        if not self.live:
            for item in self._load_board()["work_items"]:
                if item["id"] == work_item_id:
                    return item
            raise ValueError(f"Work item {work_item_id} not found")
        return self._request("GET", self._org_api(
            f"wit/workitems/{work_item_id}?$expand=relations&api-version={API_VERSION}")).json()

    def get_open_workload(self, project: str | None = None) -> dict:
        """Remaining hours of open work per assignee."""
        items = self._load_board()["work_items"] if not self.live else self._query_open_items(project)
        workload = {}
        for item in items:
            fields = item["fields"]
            person = fields.get("System.AssignedTo")
            if isinstance(person, dict):
                person = person.get("displayName")
            if person and fields.get("System.State", "New") not in CLOSED_STATES:
                hours = float(fields.get("Microsoft.VSTS.Scheduling.RemainingWork") or 0)
                workload[person] = workload.get(person, 0) + hours
        return workload

    def _query_open_items(self, project: str | None = None) -> list:
        if not (project or self.project):
            return []  # no project to look in (ADO_PROJECT not set)
        wiql = {"query": "SELECT [System.Id] FROM WorkItems WHERE [System.TeamProject] = @project "
                         "AND [System.State] NOT IN ('Closed', 'Done', 'Removed', 'Resolved') "
                         "AND [System.AssignedTo] <> ''"}
        ids = [w["id"] for w in self._request(
            "POST", self._api(f"wiql?api-version={API_VERSION}", project), json=wiql).json()["workItems"]]
        return [{"fields": f} for f in self._batch_fields(
            ids, ["System.AssignedTo", "System.State", "Microsoft.VSTS.Scheduling.RemainingWork"])]

    def _batch_fields(self, ids: list, fields: list) -> list:
        items = []
        for start in range(0, len(ids), 200):  # the batch API accepts 200 IDs per call
            response = self._request("POST", self._org_api(f"wit/workitemsbatch?api-version={API_VERSION}"),
                                     json={"ids": ids[start:start + 200], "fields": fields})
            items += [{**w["fields"], "System.Id": w["id"]} for w in response.json()["value"]]
        return items

    # ---------------- customers and their support Features ----------------
    def list_projects(self) -> list:
        if not self.live:
            return [{"name": name} for name in self._load_board().get("projects", [])]
        projects, url = [], self._org_api(f"projects?$top=500&api-version={API_VERSION}")
        response = self._request("GET", url)
        projects += [{"name": p["name"]} for p in response.json()["value"]]
        return projects

    def search_features(self, keyword: str = "support", project: str | None = None, top: int = 200) -> list:
        """Features whose title contains the keyword, newest first, in one project or the whole organization."""
        if not self.live:
            return [
                {"id": str(i["id"]), "title": i["fields"]["System.Title"], "project": i["fields"]["System.TeamProject"]}
                for i in reversed(self._load_board()["work_items"])
                if i["type"] == "Feature" and keyword.lower() in i["fields"]["System.Title"].lower()
                and (not project or i["fields"]["System.TeamProject"] == project)
            ][:top]
        keyword = keyword.replace("'", "''")  # WIQL string escaping
        where = "AND [System.TeamProject] = @project" if project else ""
        wiql = {"query": f"SELECT [System.Id] FROM WorkItems WHERE [System.WorkItemType] = 'Feature' "
                         f"AND [System.Title] CONTAINS '{keyword}' {where} ORDER BY [System.ChangedDate] DESC"}
        url = self._api(f"wiql?$top={top}&api-version={API_VERSION}", project) if project \
            else self._org_api(f"wit/wiql?$top={top}&api-version={API_VERSION}")
        ids = [w["id"] for w in self._request("POST", url, json=wiql).json()["workItems"]][:top]
        return [{"id": str(f["System.Id"]), "title": f.get("System.Title", ""), "project": f.get("System.TeamProject", "")}
                for f in self._batch_fields(ids, ["System.Title", "System.TeamProject"])]

    def match_customer(self, sender_email: str, threshold: float = 0.5) -> dict:
        """Find the customer's project from the sender's email domain: project names first, then Feature titles."""
        domain = email_domain(sender_email)
        best = max(((name_similarity(domain, p["name"]), p["name"]) for p in self.list_projects()), default=(0, None))
        if best[0] >= threshold:
            return {"domain": domain, "project": best[1], "score": round(best[0], 2), "matched_by": "project name"}
        best_feature = max(((name_similarity(domain, f["title"].split(" - ")[0]), f) for f in self.search_features()),
                           key=lambda pair: pair[0], default=(0, None))
        if best_feature[1] and best_feature[0] >= threshold:
            return {"domain": domain, "project": best_feature[1]["project"], "score": round(best_feature[0], 2),
                    "matched_by": f"feature '{best_feature[1]['title']}'"}
        return {"domain": domain, "project": None, "score": round(max(best[0], best_feature[0]), 2), "matched_by": "none"}

    def remembered_feature(self, domain: str) -> dict | None:
        if not CUSTOMER_FEATURES.exists():
            return None
        with CUSTOMER_FEATURES.open(newline="", encoding="utf-8") as handle:
            for row in csv.DictReader(handle):
                if row["domain"].lower() == domain.lower():
                    return {"id": row["feature_id"], "title": row["title"], "project": row["project"]}
        return None

    def remember_feature(self, domain: str, feature: dict) -> None:
        rows = []
        if CUSTOMER_FEATURES.exists():
            with CUSTOMER_FEATURES.open(newline="", encoding="utf-8") as handle:
                rows = [r for r in csv.DictReader(handle) if r["domain"].lower() != domain.lower()]
        rows.append({"domain": domain, "feature_id": feature["id"], "title": feature["title"],
                     "project": feature.get("project", "")})
        with CUSTOMER_FEATURES.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["domain", "feature_id", "title", "project"])
            writer.writeheader()
            writer.writerows(rows)

    def suggest_features(self, sender_email: str) -> dict:
        """Support Features for a customer email: the remembered one first, else the matched project's features."""
        domain = email_domain(sender_email)
        remembered = self.remembered_feature(domain)
        if remembered:
            return {"domain": domain, "from_memory": True, "match": None, "default": remembered, "features": [remembered]}
        match = self.match_customer(sender_email)
        features = self.search_features(project=match["project"]) if match["project"] else []
        seen = {f["id"] for f in features}
        features += [f for f in self.search_features() if f["id"] not in seen]  # the rest of the organization
        return {"domain": domain, "from_memory": False, "match": match,
                "default": features[0] if match["project"] and features else None, "features": features}

    # ---------------- attachments ----------------
    def attach_file(self, work_item_id: int, path: str, project: str, comment: str = "") -> dict:
        """Upload a file to Azure DevOps and link it to a work item."""
        file = Path(path)
        if not file.is_file():
            raise FileNotFoundError(f"Attachment not found: {path}")
        if not self.live:
            board = self._load_board()
            item = next(i for i in board["work_items"] if i["id"] == work_item_id)
            item.setdefault("relations", []).append(
                {"rel": "AttachedFile", "url": f"dry-run://attachments/{file.name}", "attributes": {"comment": comment}})
            self._save_board(board)
            return {"name": file.name, "url": f"dry-run://attachments/{file.name}"}

        upload = self._request(
            "POST", self._api(f"attachments?fileName={quote(file.name)}&api-version={API_VERSION}", project),
            data=file.read_bytes(), headers={"Content-Type": "application/octet-stream"}).json()
        self._request(
            "PATCH", self._org_api(f"wit/workitems/{work_item_id}?api-version={API_VERSION}"),
            json=[{"op": "add", "path": "/relations/-",
                   "value": {"rel": "AttachedFile", "url": upload["url"], "attributes": {"comment": comment}}}],
            headers={"Content-Type": "application/json-patch+json"})
        return {"name": file.name, "url": upload["url"]}

    def add_link(self, work_item_id: int, url: str, comment: str) -> dict:
        """Add a hyperlink (for example the SharePoint folder of the ticket) to a work item."""
        relation = {"rel": "Hyperlink", "url": url, "attributes": {"comment": comment}}
        if not self.live:
            board = self._load_board()
            item = next(i for i in board["work_items"] if i["id"] == work_item_id)
            if all(r.get("url") != url for r in item.setdefault("relations", [])):
                item["relations"].append(relation)
            self._save_board(board)
            return {"work_item_id": work_item_id, "url": url, "status": "linked"}
        existing = self.get_work_item(work_item_id).get("relations") or []
        if any(r.get("url") == url for r in existing):
            return {"work_item_id": work_item_id, "url": url, "status": "already linked"}
        self._request("PATCH", self._org_api(f"wit/workitems/{work_item_id}?api-version={API_VERSION}"),
                      json=[{"op": "add", "path": "/relations/-", "value": relation}],
                      headers={"Content-Type": "application/json-patch+json"})
        return {"work_item_id": work_item_id, "url": url, "status": "linked"}

    def attach_documents(self, work_item_id: int, paths: list, comment: str) -> list:
        """Attach generated documents (for example a Change Request) to an existing work item."""
        fields = self.get_work_item(work_item_id).get("fields", {})
        project = fields.get("System.TeamProject") or self.project or "dry-run"
        results = []
        for path in paths:
            try:
                results.append({**self.attach_file(work_item_id, path, project, comment), "status": "attached"})
            except Exception as error:
                results.append({"name": Path(path).name, "status": f"failed: {error}"})
        return results

    # ---------------- support email -> story + small tasks ----------------
    def create_support_items(self, package: dict) -> dict:
        """One User Story under the customer's support Feature, its small tasks as children (Activity, estimate,
        due date), and the email's attachments on the story. Recommended owners go in the task descriptions."""
        esc = html.escape
        feature, ticket, tasks = package["feature"], package["ticket"], package["tasks"]
        project = feature.get("project") or self.project or "dry-run"
        total_hours = sum(t["estimate_hours"] for t in tasks)
        steps = "".join(f"<li>{esc(s)}</li>" for s in ticket["steps_to_reproduce"])
        story = self.create_work_item(self.story_type, {
            "System.Title": ticket["title"],
            "System.Description": (
                f"<p>{esc(ticket['description']).replace(chr(10), '<br>')}</p>"
                + (f"<p><b>Steps to reproduce:</b></p><ol>{steps}</ol>" if steps else "")
                + f"<p><b>Customer:</b> {esc(package['customer'])} | <b>Email:</b> {esc(package['email_subject'])}</p>"
            ),
            "Microsoft.VSTS.Common.AcceptanceCriteria":
                "<ul>" + "".join(f"<li>{esc(c)}</li>" for c in ticket["acceptance_criteria"]) + "</ul>",
            "Microsoft.VSTS.Common.Priority": PRIORITY.get(ticket["priority"], 2),
            "Microsoft.VSTS.Common.Activity": package.get("story_activity", "Design"),
            "Microsoft.VSTS.Scheduling.StoryPoints": total_hours,
            "Microsoft.VSTS.Scheduling.OriginalEstimate": total_hours,
            "System.Tags": "; ".join(["Support", ticket["category"], ticket["priority"], ticket["module"]]
                                     + (["Change Request"] if package.get("cr_required") else [])),
        }, parent_id=int(feature["id"]), project=project)

        created_tasks = []
        for task in tasks:
            fields = {
                "System.Title": task["title"],
                "System.Description": (
                    f"<p>{esc(task['description'])}</p>"
                    f"<p><b>Recommended owner (to be confirmed):</b> {esc(str(task.get('owner')))} - "
                    f"{esc(task.get('owner_reason', ''))}</p>"
                ),
                "Microsoft.VSTS.Common.Activity": task["activity"],
                "Microsoft.VSTS.Scheduling.OriginalEstimate": task["estimate_hours"],
                "Microsoft.VSTS.Scheduling.RemainingWork": task["estimate_hours"],
                "System.Tags": "; ".join(["Support", task["kind"]]),
            }
            if self.due_date_field and task.get("due_date"):
                fields[self.due_date_field] = f"{task['due_date']}T12:00:00Z"
            item = self.create_work_item("Task", fields, parent_id=story["id"], project=project)
            created_tasks.append({**item, "title": task["title"], "activity": task["activity"],
                                  "estimate_hours": task["estimate_hours"], "due_date": task.get("due_date")})

        attachments = []
        for path in package.get("attachment_paths", []):
            try:
                attachments.append({**self.attach_file(story["id"], path, project,
                                                       f"From the customer email: {package['email_subject']}"),
                                    "status": "attached"})
            except Exception as error:  # report every failed file instead of hiding it
                attachments.append({"name": Path(path).name, "status": f"failed: {error}"})

        self.remember_feature(package["domain"], feature)
        return {"mode": self.mode, "project": project, "story": {**story, "title": ticket["title"]},
                "tasks": created_tasks, "attachments": attachments}

    # ---------------- delivery plan -> linked work items ----------------
    def create_delivery_items(self, request_id: str, module: str, work_type: str,
                              plan: dict, owner_recommendations: dict) -> list:
        """Create one User Story per planned story and link its FUNC/TECH tasks as children.
        Recommended owners are written into the description; System.AssignedTo is left empty."""
        esc = html.escape
        created = []
        for story in plan["user_stories"]:
            criteria = "".join(
                f"<li><b>{esc(ac['ac_id'])}</b> ({esc(', '.join(ac['req_ids']))}): Given {esc(ac['given'])}, "
                f"when {esc(ac['when'])}, then {esc(ac['then'])}</li>"
                for ac in story["acceptance_criteria"]
            )
            story_item = self.create_work_item(self.story_type, {
                "System.Title": story["title"],
                "System.Description": (
                    f"<p>As a {esc(story['as_a'])}, I want {esc(story['i_want'])} so that {esc(story['so_that'])}.</p>"
                    f"<p>Request: {esc(request_id)} | Requirements: {esc(', '.join(story['req_ids']))}</p>"
                ),
                "Microsoft.VSTS.Common.AcceptanceCriteria": f"<ul>{criteria}</ul>",
                "System.Tags": "; ".join([module, work_type, request_id, *story["req_ids"]]),
            })
            created.append({**story_item, "ref": story["story_id"], "title": story["title"]})

            for task in story["tasks"]:
                owner = owner_recommendations.get(task["task_id"], {})
                task_item = self.create_work_item("Task", {
                    "System.Title": task["title"],
                    "System.Description": (
                        f"<p>{esc(task['description'])}</p>"
                        f"<p>Requirements: {esc(', '.join(task['req_ids']))}</p>"
                        f"<p><b>Recommended owner (to be confirmed):</b> {esc(str(owner.get('owner')))} - "
                        f"{esc(owner.get('reason', ''))}</p>"
                    ),
                    "Microsoft.VSTS.Scheduling.RemainingWork": task["estimate_hours"],
                    "System.Tags": "; ".join([module, task["kind"], request_id, *task["req_ids"]]),
                }, parent_id=story_item["id"])
                created.append({**task_item, "ref": task["task_id"], "title": task["title"],
                                "parent_id": story_item["id"]})
        return created

In [ ]:
import importlib

import ado_client

importlib.reload(ado_client)
devops = ado_client.AzureDevOpsClient()
print("Azure DevOps mode:", devops.mode)
show("OPEN WORKLOAD (hours)", devops.get_open_workload())

## Owner Recommendation

For every task the recommender keeps only people with the matching role (functional consultant or developer),
prefers people specialised in the request's module, and then picks the person with the most free hours before
the deadline. Hours recommended earlier in the same plan are counted, so the work is spread across the team.

In [ ]:
from datetime import date

ROLE_FOR_TASK = {"Functional": "Functional Consultant", "Technical": "Developer"}


def recommend_owners(plan: dict, module: str, team: list, workload: dict, deadline: str) -> dict:
    """Recommend (never assign) an owner per task from role, specialization, open workload and deadline."""
    weeks_left = max((date.fromisoformat(deadline) - date.today()).days / 7, 1)
    load = dict(workload)  # includes hours already recommended in this plan
    recommendations = {}

    for story in plan["user_stories"]:
        for task in story["tasks"]:
            candidates = []
            for member in team:
                if member["role"] != ROLE_FOR_TASK[task["kind"]]:
                    continue
                specialised = any(s in module for s in member["specializations"])
                free_hours = member["capacity_hours_per_week"] * weeks_left - load.get(member["name"], 0)
                candidates.append((specialised, free_hours, member["name"]))

            if not candidates:
                recommendations[task["task_id"]] = {"owner": None, "reason": "No team member has the required role."}
                continue

            specialised, free_hours, name = max(candidates)
            load[name] = load.get(name, 0) + task["estimate_hours"]
            warning = "" if free_hours >= task["estimate_hours"] else " WARNING: not enough capacity before the deadline."
            recommendations[task["task_id"]] = {
                "owner": name,
                "reason": (f"{ROLE_FOR_TASK[task['kind']]}, "
                           f"{'specialised in ' + module if specialised else 'no ' + module + ' specialisation'}, "
                           f"about {free_hours:.0f} free hours before {deadline}.{warning}"),
            }
    return recommendations
owners = recommend_owners(plan, CLASSIFICATION["module"], TEAM, devops.get_open_workload(), DEADLINE)
for task_id, recommendation in owners.items():
    print(f"{task_id}: {recommendation['owner']:<14} {recommendation['reason']}")

## Azure DevOps MCP Server

The server exposes seven tools: `get_team_workload`, `get_work_item`, `suggest_features_for_sender`,
`create_linked_work_items`, `create_support_work_items`, `attach_documents` (a generated Change Request)
and `add_document_link` (the SharePoint folder of a ticket, only links inside the configured site). The create tools need an approval ID: the
artefact type `final_delivery_package` for the delivery plan, and `support_ticket` for that exact email for a
support ticket.

In [ ]:
%%writefile approval_ledger.py
"""Approval tool: every human decision is recorded so that MCP tools can verify approval before acting."""
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

LEDGER = Path("approvals.json")


def _load() -> list:
    return json.loads(LEDGER.read_text(encoding="utf-8")) if LEDGER.exists() else []


def record_decision(artefact_type: str, artefact_ref: str, approver: str, decision: str, comments: str = "") -> dict:
    record = {
        "approval_id": f"APR-{uuid.uuid4().hex[:8].upper()}",
        "artefact_type": artefact_type,
        "artefact_ref": artefact_ref,
        "approver": approver,
        "decision": decision,
        "comments": comments,
        "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    records = _load()
    records.append(record)
    LEDGER.write_text(json.dumps(records, indent=2), encoding="utf-8")
    return record


def is_approved(approval_id: str, artefact_type: str, artefact_ref: str | None = None) -> bool:
    """True when a person approved this artefact type (and, if given, this exact artefact)."""
    return any(
        r["approval_id"] == approval_id and r["artefact_type"] == artefact_type and r["decision"] == "approved"
        and (artefact_ref is None or r["artefact_ref"] == artefact_ref)
        for r in _load()
    )

In [ ]:
import approval_ledger

importlib.reload(approval_ledger)
Path("approvals.json").unlink(missing_ok=True)

In [ ]:
%%writefile devops_server.py
"""Azure DevOps MCP server: reads team workload, finds a customer's support Feature, and creates linked work
items (delivery plans and support tickets) after approval."""
import json
import os
import sys
from pathlib import Path

from mcp.server.fastmcp import FastMCP

from ado_client import AzureDevOpsClient
from approval_ledger import is_approved

mcp = FastMCP("azure_devops")
client = AzureDevOpsClient()


def log(message: str) -> None:
    print(f"[devops tool:{client.mode}] {message}", file=sys.stderr, flush=True)


@mcp.tool()
def get_team_workload(project: str = "") -> str:
    """Return the remaining hours of open work per assignee in an Azure DevOps project (default project if empty)."""
    log(f"get_team_workload({project!r})")
    return json.dumps(client.get_open_workload(project or None))


@mcp.tool()
def get_work_item(work_item_id: int) -> str:
    """Return one work item with its fields and links."""
    log(f"get_work_item({work_item_id})")
    return json.dumps(client.get_work_item(work_item_id))


@mcp.tool()
def suggest_features_for_sender(sender_email: str) -> str:
    """Find the customer's project and support Features from the sender's email domain.
    A Feature chosen before for the same domain is returned first (from_memory = true)."""
    log(f"suggest_features_for_sender({sender_email!r})")
    return json.dumps(client.suggest_features(sender_email))


@mcp.tool()
def create_linked_work_items(package_json: str, approval_id: str) -> str:
    """Create User Stories with acceptance criteria and linked FUNC/TECH child tasks.
    package_json: JSON with request_id, module, work_type, plan and owner_recommendations.
    approval_id: an approved 'final_delivery_package' decision from the approval ledger."""
    log(f"create_linked_work_items(approval_id={approval_id!r})")
    if not is_approved(approval_id, "final_delivery_package"):
        raise PermissionError("The delivery package has not been approved by a person.")

    package = json.loads(package_json)
    created = client.create_delivery_items(
        package["request_id"], package["module"], package["work_type"],
        package["plan"], package.get("owner_recommendations", {}),
    )
    return json.dumps({"mode": client.mode, "created": created})


@mcp.tool()
def create_support_work_items(package_json: str, approval_id: str) -> str:
    """Create a support ticket: one User Story under the customer's Feature, its small tasks as children
    (Activity, estimate, due date, recommended owner) and the email's attachments on the story.
    The Feature is remembered for the sender's domain.
    package_json: JSON with message_id, domain, customer, email_subject, feature, ticket, tasks, attachment_paths.
    approval_id: an approved 'support_ticket' decision for this message_id from the approval ledger."""
    package = json.loads(package_json)
    log(f"create_support_work_items(feature={package['feature']['id']}, approval_id={approval_id!r})")
    if not is_approved(approval_id, "support_ticket", package["message_id"]):
        raise PermissionError("The support ticket has not been approved by a person.")
    return json.dumps(client.create_support_items(package))


@mcp.tool()
def attach_documents(work_item_id: int, file_paths: list[str], message_id: str, approval_id: str) -> str:
    """Attach documents generated in output/ (for example the Change Request for customer sign-off) to a
    support work item. approval_id: the approved 'support_ticket' decision for message_id."""
    log(f"attach_documents(work_item_id={work_item_id}, files={len(file_paths)}, approval_id={approval_id!r})")
    if not is_approved(approval_id, "support_ticket", message_id):
        raise PermissionError("The support ticket has not been approved by a person.")
    outside = [p for p in file_paths if not Path(p).resolve().is_relative_to(Path("output").resolve())]
    if outside:
        raise PermissionError(f"Only documents generated in output/ can be attached: {outside}")
    return json.dumps(client.attach_documents(work_item_id, file_paths, "Change Request for customer sign-off"))


@mcp.tool()
def add_document_link(work_item_id: int, url: str, approval_id: str, message_id: str = "") -> str:
    """Link the SharePoint folder of a ticket or request to a work item.
    url: must be inside SHAREPOINT_SITE_URL (or the local dry-run folder).
    approval_id: the approved 'support_ticket' decision for message_id, or the approved 'final_delivery_package'."""
    log(f"add_document_link(work_item_id={work_item_id}, approval_id={approval_id!r})")
    if not (is_approved(approval_id, "support_ticket", message_id or None) or is_approved(approval_id, "final_delivery_package")):
        raise PermissionError("Linking needs an approved support ticket or delivery package.")
    site = os.getenv("SHAREPOINT_SITE_URL", "").rstrip("/")
    allowed = (site and url.startswith(site + "/")) or url.startswith(Path("sharepoint_dry_run").resolve().as_uri())
    if not allowed:
        raise PermissionError(f"Only links to the SharePoint site {site or '(not configured)'} can be added.")
    return json.dumps(client.add_link(work_item_id, url, "Documents in SharePoint"))


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
import asyncio
import contextlib
import contextvars
import json
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_LOG = Path("mcp_server.log")  # the servers' stderr ([tool called] lines)


def mcp_server(script: str) -> dict:
    return {"script": str(Path(script).resolve())}


@contextlib.asynccontextmanager
async def mcp_session(server: str):
    """Start a local FastMCP server over stdio and open an MCP session with it.

    The server's stderr goes to mcp_server.log: a notebook's own stderr cannot be handed to a subprocess.
    The whole environment is passed on (MCP only forwards a few safe variables by default), so the servers
    receive the Ollama and Azure DevOps settings.
    """
    params = StdioServerParameters(
        command=sys.executable, args=[MCP_SERVERS[server]["script"]], cwd=str(Path.cwd()), env=dict(os.environ)
    )
    with open(MCP_LOG, "a", encoding="utf-8") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


def run_async(coro):
    """Run a coroutine from a notebook cell and print the servers' tool log lines.

    It uses a fresh event loop in a worker thread, so it also works in Jupyter on Windows,
    where the notebook's own loop cannot start subprocesses.
    """
    def _run():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        try:
            return loop.run_until_complete(coro)
        finally:
            loop.close()

    log_start = MCP_LOG.stat().st_size if MCP_LOG.exists() else 0
    context = contextvars.copy_context()  # keeps MCP tool calls inside the current LangSmith trace
    with ThreadPoolExecutor(max_workers=1) as pool:
        result = pool.submit(context.run, _run).result()

    if MCP_LOG.exists():
        with open(MCP_LOG, encoding="utf-8") as log:
            log.seek(log_start)
            for line in log.read().splitlines():
                if line.startswith("[") and "tool" in line.split("]")[0]:
                    print(line)
    return result


def _to_python(result):
    if isinstance(result, list):  # recent adapters return a list of content blocks
        result = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in result)
    try:
        return json.loads(result)
    except (TypeError, json.JSONDecodeError):
        return result


def call_mcp_tools(server: str, calls: list) -> list:
    """Open one MCP session and run several (tool_name, arguments) calls in it."""
    async def _calls():
        async with mcp_session(server) as session:
            tools = {t.name: t for t in await load_mcp_tools(session)}
            results = []
            for name, args in calls:
                try:
                    results.append(_to_python(await tools[name].ainvoke(args)))
                except Exception as error:
                    print(f"[mcp error] {server}.{name}: {error}")
                    results.append({"error": str(error)})
            return results

    return run_async(_calls())


def call_mcp_tool(server: str, tool_name: str, **arguments):
    return call_mcp_tools(server, [(tool_name, arguments)])[0]


def list_mcp_tools(server: str) -> None:
    """tools/list: print the tools a server exposes, with their descriptions and inputs."""
    async def _list():
        async with mcp_session(server) as session:
            return (await session.list_tools()).tools

    print(f"========== TOOLS DISCOVERED ON '{server}' ==========\n")
    for tool in run_async(_list()):
        print(f"Name: {tool.name}")
        print(f"Description: {' '.join(tool.description.split())}")
        print(f"Inputs: {list(tool.inputSchema.get('properties', {}))}\n")
MCP_SERVERS = {"devops": mcp_server("devops_server.py")}

list_mcp_tools("devops")

## Guardrail Test: Creating Work Items Without Approval

In [ ]:
package = {
    "request_id": REQUEST_ID,
    "module": CLASSIFICATION["module"],
    "work_type": CLASSIFICATION["work_type"],
    "plan": plan,
    "owner_recommendations": owners,
}

print(call_mcp_tool("devops", "create_linked_work_items", package_json=json.dumps(package), approval_id="APR-FAKE0001"))

## Human Approval, Then Create the Linked Work Items

In [ ]:
# "auto" = scripted demo decisions (always approve, simulated customer reply)
# "interactive" = you type every decision and the customer's reply yourself
HUMAN_MODE = "auto"
APPROVER = "Toni Kanaan"


def human_decision(title: str, artefact: str, options=("approve", "revise", "reject"), live_action: bool = False) -> dict:
    """Show an artefact to a person and return their decision and comments.
    live_action: approving triggers a real action (real work items, a real email). Auto mode never approves it."""
    print(f"\n==================== HUMAN REVIEW: {title} ====================\n")
    print(artefact)
    print()

    if HUMAN_MODE == "interactive":
        decision = ""
        while decision not in options:
            decision = input(f"Decision {options}: ").strip().lower()
        comments = input("Comments (optional): ").strip()
    elif live_action:
        decision, comments = "reject", "Auto mode never approves a live action. Set HUMAN_MODE = 'interactive'."
        print(f"[auto] decision = {decision}: {comments}")
    else:
        decision, comments = "approve", "Approved in auto demo mode."
        print(f"[auto] decision = {decision}")

    return {"decision": decision, "comments": comments}
review_text = "\n".join(
    f"{s['story_id']} {s['title']} - {len(s['acceptance_criteria'])} criteria, "
    + ", ".join(f"{t['task_id']} -> {owners[t['task_id']]['owner']}" for t in s["tasks"])
    for s in plan["user_stories"]
)
review = human_decision("Work items to create in Azure DevOps", review_text, options=("approve", "reject"),
                        live_action=devops.live)
record = approval_ledger.record_decision(
    "final_delivery_package", REQUEST_ID, APPROVER,
    "approved" if review["decision"] == "approve" else "rejected", review["comments"],
)

result = None
if record["decision"] == "approved":
    result = call_mcp_tool("devops", "create_linked_work_items",
                           package_json=json.dumps(package), approval_id=record["approval_id"])
    print(f"Mode: {result['mode']}\n")
    for item in result["created"]:
        indent = "      " if item.get("parent_id") else ""
        print(f"{indent}#{item['id']} {item['type']:<11} {item['title']}")
else:
    print("Rejected: no work items were created.")

### Check one created task

The task is linked to its story, carries the estimate and the recommended owner, and has no assignee.

In [ ]:
if result and result.get("created"):
    first_task = next(item for item in result["created"] if item["type"] == "Task")
    task = call_mcp_tool("devops", "get_work_item", work_item_id=first_task["id"])
    show("CREATED TASK", task)
    print("Assigned to:", task.get("fields", {}).get("System.AssignedTo", "nobody (recommendation only)"))
else:
    print("No work items were created.")

## Support Emails to Work Items

The same building blocks handle day-to-day support. One support email goes through these steps:

1. **Read** the email and its attachments through the email MCP server from notebook 03 (Outlook through
   Microsoft Graph, or the local mailbox).
2. **Find the customer's support Feature.** A Feature chosen before for the sender's domain is reused.
   Otherwise the domain is matched to a project name (or to a Feature title), and that project's support
   Features are proposed first.
3. The **Support Planning Agent** writes the ticket (title, description, priority, category, acceptance
   criteria, steps to reproduce) and **splits the work into small tasks** of at most 8 hours, each with an
   Activity and an estimate.
4. **Due dates** are scheduled from the estimates (business days, `ADO_HOURS_PER_DAY` hours a day, default 6)
   and an **owner is recommended** for each task.
5. After approval: a **User Story under the Feature**, the tasks as its children (Activity, Original Estimate,
   Remaining Work, due date) and the **attachments on the story**. The Feature is remembered for the domain.
6. The **acknowledgement email** with the ticket number is drafted, approved, saved as a draft and **sent**
   after a second approval.

In [ ]:
SAMPLE_SUPPORT_EMAIL = {  # used when the email server of notebook 03 is not available
    "message_id": "SAMPLE-1", "subject": "F&O support - Sales order confirmation fails",
    "from_name": "Omar Saad", "from_address": "omar.saad@cedarretail.example", "attachments": [],
    "body": ("Since this morning, sales orders in USMF cannot be confirmed. When we click \"Confirm sales order\" "
             "we get the message \"Update has been canceled\" and the order stays open. This blocks today's "
             "shipments.\n\nThanks,\nOmar Saad\nSales Operations, Cedar Retail Group"),
}

EMAIL_SERVER_READY = Path("email_server.py").exists() and Path("mail_client.py").exists()
if EMAIL_SERVER_READY:
    MCP_SERVERS["email"] = mcp_server("email_server.py")
    inbox = call_mcp_tool("email", "list_support_emails", max_emails=10)
    if not inbox:
        raise RuntimeError("No support emails found: check the mailbox folder and the support filter in .env.")
    for number, item in enumerate(inbox):
        print(f"{number}: {item['subject']}  ({item['from']}){'  [attachments]' if item['attachments'] else ''}")
    choice = int(input("Email number: ") or 0) if HUMAN_MODE == "interactive" else 0
    support_email = call_mcp_tool("email", "get_email", message_id=inbox[choice]["message_id"])
else:
    print("Run notebook 03 first to build the email MCP server. Using a sample email (no attachments, no reply).")
    support_email = SAMPLE_SUPPORT_EMAIL

print(f"\nFrom: {support_email['from_name']} <{support_email['from_address']}>")
print(f"Subject: {support_email['subject']}")
print(f"Attachments: {', '.join(a['name'] for a in support_email['attachments']) or 'none'}\n")
print(support_email["body"])

In [ ]:
suggestion = call_mcp_tool("devops", "suggest_features_for_sender", sender_email=support_email["from_address"])
if suggestion["from_memory"]:
    print(f"Feature remembered for '{suggestion['domain']}'")
else:
    print("Customer match:", suggestion["match"])

features = suggestion["features"]
for number, item in enumerate(features[:15]):
    print(f"{number}: #{item['id']} {item['title']}  [{item['project']}]")

if HUMAN_MODE == "interactive":
    feature = features[int(input("Feature number: ") or 0)]
else:
    feature = suggestion["default"]
if feature is None:
    raise ValueError("No support Feature matches this customer. Choose one with HUMAN_MODE = 'interactive'.")

customer = feature["title"].split(" - ")[0]
print(f"\nLinked Feature: #{feature['id']} {feature['title']} (customer: {customer}, project: {feature['project']})")

### Support Planning Agent

The agent returns the ticket and its tasks. Tasks larger than 8 hours go back to the agent once with that
feedback. The ticket title is prefixed with the customer name (`Customer - short task`), as in the
EmailToWorkItem app.

In [ ]:
from typing import Literal

MAX_TASK_HOURS = 8


class SupportTask(BaseModel):
    title: str = Field(description="Starts with FUNC: or TECH:, short and specific")
    description: str = Field(description="What to do, in one or two sentences")
    kind: Literal["Functional", "Technical"]
    activity: Literal["Requirements", "Design", "Development", "Testing", "Documentation", "Deployment"]
    estimate_hours: float = Field(description=f"Realistic estimate between 0.5 and {MAX_TASK_HOURS} hours")


class SupportTicketPlan(BaseModel):
    title: str = Field(description="The task only, without company or sender, e.g. 'Fix sales order confirmation error'")
    description: str = Field(description="The problem as bullet points, only facts from the email; mention attached files")
    module: Literal["Finance", "Supply Chain", "Finance and Supply Chain", "Out of scope"]
    priority: Literal["Low", "Medium", "High"]
    category: Literal["Bug", "Support", "Feature Request"]
    acceptance_criteria: list[str]
    steps_to_reproduce: list[str] = Field(description="Steps given in the email; empty when there are none")
    tasks: list[SupportTask]


SUPPORT_PLANNING_SYSTEM = f"""
You are the Support Planning Agent of a Dynamics 365 F&O support team.
Turn a customer support email into one Azure DevOps ticket and split the work into small tasks.

Rules:
- Use only facts from the email. Write "Not specified" for missing information and never invent values.
- The title is the task itself, without the company or the sender's name.
- Split the work into small, concrete tasks in the order they are done: investigate and reproduce, fix or
  configure, test, then document and inform the customer. Each task has one Activity and takes at most
  {MAX_TASK_HOURS} hours; split bigger work into several tasks.
- Activity: Requirements = clarifying the need with the customer, Design = solution or configuration design,
  Development = investigating, reproducing and fixing (code, configuration, data), Testing = verifying the
  fix, Documentation = documenting and informing the customer, Deployment = moving the fix to production.
- Functional tasks (analysis, configuration, testing with users) start with "FUNC:"; technical tasks
  (X++, integrations, data fixes, deployment) start with "TECH:".
- Priority is High only when the customer's business is blocked.
"""


@traceable(name="Support Planning Agent")
def plan_support_ticket(email: dict, feedback: str = "") -> dict:
    attachments = ", ".join(a["name"] for a in email["attachments"]) or "None"
    user_prompt = (f"From: {email['from_name']} <{email['from_address']}>\nSubject: {email['subject']}\n"
                   f"Attachments: {attachments}\n\n{email['body']}")
    if feedback:
        user_prompt += f"\n\nFix this problem in your plan: {feedback}"
    return ask_structured(SupportTicketPlan, SUPPORT_PLANNING_SYSTEM, user_prompt).model_dump()


def ticket_title(customer: str, title: str) -> str:
    """'Customer - short task', without repeating the customer name the model may have added."""
    title = title.strip()
    while title.lower().startswith(customer.lower() + " - "):
        title = title[len(customer) + 3:].strip()
    return f"{customer} - {title or 'Email-based support request'}"


ticket = plan_support_ticket(support_email)
too_big = [t["title"] for t in ticket["tasks"] if t["estimate_hours"] > MAX_TASK_HOURS]
if too_big or not ticket["tasks"]:
    print("Tasks to split:", too_big or "no tasks", "-> sending back to the Support Planning Agent")
    ticket = plan_support_ticket(support_email, feedback=(
        f"Split these tasks into tasks of at most {MAX_TASK_HOURS} hours: {too_big}" if too_big else "Add the tasks."))
ticket["title"] = ticket_title(customer, ticket["title"])
show("SUPPORT TICKET", {k: v for k, v in ticket.items() if k != "tasks"})

### Does This Email Need a Change Request?

The **CR Assessment Agent** applies the company CR standard (`STD-CR-01`): a defect in delivered functionality is
support; new or changed functionality, customization, reports or scope need a **Change Request** signed by the
customer. When a CR is needed, the **Change Request Agent** fills the company's *ERP Change/Additional Request
Form* after the ticket exists (its change number is the User Story ID); the form is attached to the story and to
the acknowledgement email.

In [ ]:
# Retrieved from the knowledge base (STD-CR-01)
CR_STANDARD = "A Change Request (CR) is required before any work that changes the agreed solution: new or changed functionality, customizations (X++ extensions), new or changed reports, integrations, data entities, workflows or fields, changes to the agreed scope, legal entities or business processes, and any configuration change outside the support contract. No CR is needed for support: defects in functionality that was delivered and accepted, errors caused by data or setup that are restored to the agreed design, how-to questions, access or security issues within the agreed roles, and standard Microsoft updates. When a request mixes both, the support part is handled as a ticket and the changed part as a CR. A CR uses the ERP Change/Additional Request Form: project name, change number, requested by, date of request, required phase, business priority, change name and technical complexity; the change description (business requirement and proposed solution) and the recommendation/best practice (customization required, affected areas, estimated effort) are written by the implementation partner; the business justification and the impact of not implementing the change are left for the customer's business to fill in; sign-off by the customer's functional lead, head of department and IT manager and by the partner's project manager. Work on a change starts only after the CR is signed."


class CRAssessment(BaseModel):
    request_type: Literal["Support", "Change request", "Support and change request"]
    cr_required: bool = Field(description="True when any part of the request needs a Change Request")
    reasons: list[str] = Field(description="Why, each reason naming the rule of the CR standard that applies")
    change_items: list[str] = Field(description="Parts that change the agreed solution (empty when none)")
    support_items: list[str] = Field(description="Parts that are support: defects, data or setup fixes, questions")


class ChangeRequestContent(BaseModel):
    change_name: str = Field(description="Short name of the change, e.g. 'Vendor invoice approval above 10,000 USD'")
    business_requirement: str = Field(description="What the customer needs and why, in business language (1-3 short paragraphs)")
    proposed_solution: str = Field(description="How it will be delivered in Dynamics 365 F&O, in business language")
    customization_required: bool
    technical_complexity: Literal["Low", "Medium", "High"]
    business_priority: Literal["Low", "Medium", "High"] = Field(description="From the customer's urgency and impact")
    recommendation: str = Field(description="Recommendation and best practice: standard features first, customization only where needed")
    affected_areas: list[str] = Field(description="Forms, journals, workflows, reports, entities or processes affected")
    assumptions: list[str]


CR_ASSESSMENT_SYSTEM = """
You are the Change Request Assessment Agent of a Dynamics 365 F&O partner.
Decide whether a customer request needs a Change Request (CR), using only the company CR standard you are given.
- A defect in functionality that already works as agreed, a data or setup problem, a how-to question or an
  access issue is support. New or changed functionality, customizations, reports, integrations, workflows,
  fields, processes or scope are changes.
- Name the rule of the standard that applies in each reason.
- When the request mixes both, list each part and set cr_required to true.
"""

CR_WRITER_SYSTEM = """
You are the Change Request Agent of a Dynamics 365 F&O implementation partner.
Write the partner's parts of the ERP Change/Additional Request Form for the customer to review and sign.
- Business language for the customer's managers; technical object names only in affected_areas.
- Use only facts from the request, the confirmed facts and the plan. Never invent amounts, dates or names.
- Recommend standard features first and say clearly when customization is required and why.
- Do not mention prices. The estimated effort is added from the plan.
"""


@traceable(name="CR Assessment Agent")
def assess_change_request(request_text: str, context: str = "") -> CRAssessment:
    user_prompt = f"Company CR standard:\n{CR_STANDARD}\n\nCustomer request:\n{request_text}"
    if context:
        user_prompt += f"\n\nContext:\n{context}"
    return ask_structured(CRAssessment, CR_ASSESSMENT_SYSTEM, user_prompt)


@traceable(name="Change Request Agent")
def write_change_request(request_text: str, change_items: list, context: str = "") -> ChangeRequestContent:
    user_prompt = (f"Company CR standard:\n{CR_STANDARD}\n\nCustomer request:\n{request_text}\n\n"
                   "Parts that are changes:\n" + "\n".join(f"- {item}" for item in change_items))
    if context:
        user_prompt += f"\n\nConfirmed facts and plan:\n{context}"
    return ask_structured(ChangeRequestContent, CR_WRITER_SYSTEM, user_prompt)


import math
import re
from datetime import date, datetime

import docx
from docx.shared import Pt

CR_PROJECT_NAME = os.getenv("CR_PROJECT_NAME", "Dynamics 365 F&O")
CR_SIGN_OFF = ["Functional Lead - Customer", "Head of Department - Customer", "IT Manager - Customer",
               "Project Manager - Partner"]


def form_date(received: str = "") -> str:
    """'2026-09-25T09:14:00Z' -> '25/09/2026', the form's date format (today when the date is unknown)."""
    try:
        return datetime.fromisoformat(received.replace("Z", "+00:00")).strftime("%d/%m/%Y")
    except ValueError:
        return date.today().strftime("%d/%m/%Y")


def cr_file_name(change_number) -> str:
    number = str(change_number)
    return re.sub(r"[^A-Za-z0-9_-]", "_", number if not number.isdigit() else f"CR-{number}") + "_Change_Request.docx"


def export_change_request(cr: dict, meta: dict, folder: Path) -> Path:
    """Document tool: the ERP Change/Additional Request Form (the company layout) as a Word file.
    meta: change_number, requested_by, date_of_request, phase, effort_hours."""
    document = docx.Document()
    document.styles["Normal"].font.name = "Arial"
    document.styles["Normal"].font.size = Pt(10)
    document.add_paragraph("ERP Change/Additional Request Form", style="Title")

    header = document.add_table(rows=4, cols=4)
    header.style = "Table Grid"
    values = [("Project Name", CR_PROJECT_NAME, "Change Number", meta["change_number"]),
              ("Requested By", meta["requested_by"], "Date of Request", meta["date_of_request"]),
              ("Required Phase of Project", meta["phase"], "Business Priority (High, Medium, Low)", cr["business_priority"]),
              ("Change Name", cr["change_name"], "Technical Complexity (High, Medium, Low)", cr["technical_complexity"])]
    for row, texts in zip(header.rows, values):
        for column, (cell, text) in enumerate(zip(row.cells, texts)):
            cell.paragraphs[0].add_run(str(text)).bold = column % 2 == 0  # labels in bold

    def section(title: str, blocks=()) -> None:
        """One form section: a bold title row and a body row. blocks: (kind, text) with kind heading/text/bullet."""
        document.add_paragraph()
        table = document.add_table(rows=2, cols=1)
        table.style = "Table Grid"
        table.rows[0].cells[0].paragraphs[0].add_run(title).bold = True
        body = table.rows[1].cells[0]
        for number, (kind, text) in enumerate(blocks):
            paragraph = body.paragraphs[0] if number == 0 else body.add_paragraph()
            run = paragraph.add_run(f"\u2022  {text}" if kind == "bullet" else text)
            run.bold = kind == "heading"

    def paragraphs(text: str) -> list:
        return [("text", line.strip()) for line in text.split("\n") if line.strip()]

    section("Change Description: (To be filled by the Partner)",
            [("heading", "Business Requirement"), *paragraphs(cr["business_requirement"]),
             ("heading", "Proposed Solution"), *paragraphs(cr["proposed_solution"])])
    section("Business Justification (Value): (Risk, productivity, efficiency, financial uplift) "
            "(To be filled by Customer Business)")
    section("Impact of not implementing the Change: (To be filled by Customer Business)")
    effort = meta["effort_hours"]
    section("Recommendation/Best Practice: (To be filled by the Partner)",
            [("heading", f"Customization Required: {'Yes' if cr['customization_required'] else 'No'}"),
             *paragraphs(cr["recommendation"]),
             ("heading", "Affected Areas"), *[("bullet", area) for area in cr["affected_areas"]],
             ("heading", "Estimated Effort"),
             ("text", f"{effort:g} hours (about {math.ceil(effort / 8)} working days), from the delivery plan"),
             *([("heading", "Assumptions")] + [("bullet", a) for a in cr["assumptions"]] if cr["assumptions"] else [])])

    document.add_paragraph()
    approval = document.add_table(rows=1 + len(CR_SIGN_OFF), cols=1)
    approval.style = "Table Grid"
    approval.rows[0].cells[0].paragraphs[0].add_run("For Approval:").bold = True
    for row, role in zip(approval.rows[1:], CR_SIGN_OFF):
        row.cells[0].text = f"Sign off by: ______________________ ({role})"

    folder.mkdir(parents=True, exist_ok=True)
    path = folder / cr_file_name(meta["change_number"])
    document.save(path)
    return path


CR_PARAGRAPH = {
    "Change request": "This request changes the agreed solution, so it needs a Change Request.",
    "Support and change request": "Part of this request changes the agreed solution, so that part needs a Change Request.",
}


def cr_reply_paragraph(cr: dict | None) -> str:
    """The acknowledgement paragraph that introduces the attached Change Request ("" when none is needed)."""
    if not cr:
        return ""
    return (f"**Change Request:**\n{CR_PARAGRAPH.get(cr['request_type'], CR_PARAGRAPH['Change request'])} "
            f"Please review the attached Change Request **{cr['number']}**, complete the business justification "
            "and impact sections, and return it signed. Work on the change starts once it is approved.\n\n")

In [ ]:
cr_assessment = assess_change_request(
    f"Subject: {support_email['subject']}\n\n{support_email['body']}",
    context=f"Ticket category from the Support Planning Agent: {ticket['category']}",
)
print(f"{cr_assessment.request_type} -> CR {'REQUIRED' if cr_assessment.cr_required else 'not needed'}")
for reason in cr_assessment.reasons:
    print("   reason:", reason)

### Due Dates and Recommended Owners

The tasks are done one after the other: each due date is the business day on which its work is finished,
counting `ADO_HOURS_PER_DAY` hours a day. Owners are recommended with the same role, specialization and
workload rules as the delivery plan (`TEAM` is the sample roster: replace it with your team).

In [ ]:
import math
from datetime import date, timedelta

HOURS_PER_DAY = float(os.getenv("ADO_HOURS_PER_DAY", "6"))


def add_business_days(day: date, days: int) -> date:
    while day.weekday() >= 5:  # start on a working day
        day += timedelta(days=1)
    for _ in range(days):
        day += timedelta(days=1)
        while day.weekday() >= 5:
            day += timedelta(days=1)
    return day


def schedule_tasks(tasks: list, start: date) -> list:
    done, scheduled = 0.0, []
    for number, task in enumerate(tasks, start=1):
        done += task["estimate_hours"]
        due = add_business_days(start, max(math.ceil(done / HOURS_PER_DAY) - 1, 0))
        scheduled.append({**task, "task_id": f"T-{number:02d}", "due_date": due.isoformat()})
    return scheduled


support_tasks = schedule_tasks(ticket["tasks"], date.today())
workload = call_mcp_tool("devops", "get_team_workload", project=feature["project"] if devops.live else "")
support_owners = recommend_owners({"user_stories": [{"tasks": support_tasks}]}, ticket["module"], TEAM,
                                  workload, support_tasks[-1]["due_date"])
for task in support_tasks:
    task["owner"] = support_owners[task["task_id"]]["owner"]
    task["owner_reason"] = support_owners[task["task_id"]]["reason"]

print(f"{ticket['title']}  [{ticket['priority']}, {ticket['category']}, {ticket['module']}]\n")
for task in support_tasks:
    print(f"{task['task_id']} {task['title'][:60]:<60} {task['activity']:<13} {task['estimate_hours']:>5}h  "
          f"due {task['due_date']}  -> {task['owner']}")
print(f"\nTotal: {sum(t['estimate_hours'] for t in support_tasks)} hours")

### Human Approval, Then Create the Ticket

The reviewer sees the Feature, the story, every task with its Activity, estimate, due date and recommended
owner, and the attachments. `create_support_work_items` refuses to run without an approval for this email.

In [ ]:
attachment_files = []
if EMAIL_SERVER_READY and support_email["attachments"]:
    attachment_files = call_mcp_tool("email", "save_attachments", message_id=support_email["message_id"])
    if isinstance(attachment_files, dict):  # tool error: continue without the files
        print("Attachments not downloaded:", attachment_files["error"])
        attachment_files = []

support_package = {
    "message_id": support_email["message_id"],
    "domain": suggestion["domain"],
    "customer": customer,
    "email_subject": support_email["subject"],
    "feature": feature,
    "ticket": ticket,
    "tasks": support_tasks,
    "attachment_paths": [f["path"] for f in attachment_files],
    "cr_required": cr_assessment.cr_required,
}

review_text = (
    f"Feature: #{feature['id']} {feature['title']} ({feature['project']})\n"
    f"User Story: {ticket['title']}  [{ticket['priority']}, {ticket['category']}]\n"
    + "\n".join(f"  {t['task_id']} {t['title']} | {t['activity']} | {t['estimate_hours']}h | due {t['due_date']} | "
                f"{t['owner']}" for t in support_tasks)
    + f"\nAttachments: {', '.join(f['name'] for f in attachment_files) or 'none'}"
    + f"\nChange Request: {'REQUIRED (' + cr_assessment.request_type + ')' if cr_assessment.cr_required else 'not needed'}"
)
review = human_decision("Support ticket to create in Azure DevOps", review_text, options=("approve", "reject"),
                        live_action=devops.live)
record = approval_ledger.record_decision(
    "support_ticket", support_email["message_id"], APPROVER,
    "approved" if review["decision"] == "approve" else "rejected", review["comments"],
)

support_ticket = None
ticket_approval_id = record["approval_id"]
if record["decision"] == "approved":
    support_ticket = call_mcp_tool("devops", "create_support_work_items",
                                   package_json=json.dumps(support_package), approval_id=record["approval_id"])
    if "error" in support_ticket:
        raise RuntimeError(support_ticket["error"])
    story = support_ticket["story"]
    print(f"Mode: {support_ticket['mode']} | project: {support_ticket['project']}\n")
    print(f"#{story['id']} User Story  {story['title']}  (child of Feature #{feature['id']})")
    for task in support_ticket["tasks"]:
        print(f"      #{task['id']} Task  {task['title']} | {task['activity']} | {task['estimate_hours']}h | due {task['due_date']}")
    for attachment in support_ticket["attachments"]:
        print(f"      attachment {attachment['name']}: {attachment['status']}")

    again = call_mcp_tool("devops", "suggest_features_for_sender", sender_email=support_email["from_address"])
    print(f"\nNext email from '{again['domain']}' -> #{again['default']['id']} {again['default']['title']}"
          f"{' (remembered)' if again['from_memory'] else ''}")
else:
    print("Rejected: no work items were created.")

### The Change Request Document

Only when the assessment requires a CR: the form is written, exported to `output/support/<story ID>/` and attached
to the User Story with `attach_documents` (it checks the ticket approval and accepts only files from `output/`).

In [ ]:
change_request = None
if support_ticket and cr_assessment.cr_required:
    story_id = support_ticket["story"]["id"]
    cr_content = write_change_request(
        f"Subject: {support_email['subject']}\n\n{support_email['body']}", cr_assessment.change_items,
        f"Ticket: {ticket['title']}\n{ticket['description']}\n\nPlanned tasks:\n"
        + "\n".join(f"- {t['title']} ({t['estimate_hours']}h)" for t in support_tasks),
    )
    cr_path = export_change_request(cr_content.model_dump(), {
        "change_number": story_id, "requested_by": f"{support_email['from_name']} ({customer})",
        "date_of_request": form_date(support_email.get("received", "")), "phase": os.getenv("CR_SUPPORT_PHASE", "Support"),
        "effort_hours": sum(t["estimate_hours"] for t in support_tasks),
    }, Path("output") / "support" / str(story_id))
    attached = call_mcp_tool("devops", "attach_documents", work_item_id=story_id, file_paths=[str(cr_path.resolve())],
                             message_id=support_email["message_id"], approval_id=ticket_approval_id)
    change_request = {"number": f"CR-{story_id}", "request_type": cr_assessment.request_type,
                      "path": str(cr_path.resolve()), "name": cr_content.change_name}
    print(f"Saved {cr_path} -> attached to #{story_id}: {attached}")
else:
    print("No Change Request needed for this email." if support_ticket else "No ticket was created.")

### SharePoint: File the Ticket's Documents

Documents are filed in the support library on SharePoint as

**`<SHAREPOINT_ROOT_FOLDER>/<Customer>/<ID - Description>/<Document type>/<file>`**

with the document types `Change Request`, `Design`, `Test` and `Customer Files`. The site, library and root folder
come from `.env` (`SHAREPOINT_SITE_URL`, `SHAREPOINT_LIBRARY`, `SHAREPOINT_ROOT_FOLDER`); folder names are cleaned
of the characters SharePoint does not allow and kept short. An existing file with the same name gets a new
version (SharePoint keeps the history), so a signed CR can replace the draft.

Until SharePoint and Graph are configured, the same tree is written to `sharepoint_dry_run/` (dry-run). Live
access uses the app registration of notebook 03: delegated with `Sites.ReadWrite.All` (device-code sign-in), or
app-only with `GRAPH_CLIENT_SECRET` and the `Sites.Selected` permission granted on this site only. Only files
generated in `output/` or downloaded from a support email in `attachments/` can be uploaded.

The **SharePoint MCP server** has two tools: `file_documents` and `ticket_folder_url`. After filing, the ticket
folder is linked on the User Story with `add_document_link`.

In [ ]:
%%writefile sharepoint_client.py
"""SharePoint document filing through Microsoft Graph (live) or a local folder (dry-run).

Documents are filed as  <SHAREPOINT_ROOT_FOLDER>/<Customer>/<ID - Description>/<Document type>/<file>
in the library SHAREPOINT_LIBRARY of the site SHAREPOINT_SITE_URL.

Live mode needs SHAREPOINT_SITE_URL and Graph access (the app registration of notebook 03):
- Delegated (default): MailClient-style device-code sign-in with Sites.ReadWrite.All, cached in
  .graph_token_cache.json. Run SharePointClient().sign_in() once in the notebook.
- App-only: GRAPH_CLIENT_SECRET with the Sites.Selected permission, granted by an admin on this site only.
Only files generated in output/ or downloaded from a support email in attachments/ can be uploaded.
"""
import os
import re
import shutil
from pathlib import Path
from urllib.parse import quote, urlparse

import requests

GRAPH_URL = os.getenv("GRAPH_BASE_URL", "https://graph.microsoft.com/v1.0")
SCOPES = ["Sites.ReadWrite.All"]
TOKEN_CACHE = Path(".graph_token_cache.json")
DRY_RUN_DIR = Path("sharepoint_dry_run")
UPLOADABLE_DIRS = (Path("output"), Path("attachments"))
DOC_TYPES = ("Change Request", "Design", "Test", "Customer Files")
MAX_SIMPLE_UPLOAD_BYTES = 250 * 1024 * 1024


def safe_segment(text: str, max_length: int = 60) -> str:
    """One folder name SharePoint accepts: no " * : < > ? / \\ | # %, no leading/trailing dots or spaces."""
    text = re.sub(r'["*:<>?/\\|#%\x00-\x1f]', " ", str(text))
    text = re.sub(r"\s+", " ", text).strip(" .")
    if len(text) > max_length:  # cut at a word, and drop a dangling "and", "the", ... at the end
        text = text[:max_length].rsplit(" ", 1)[0]
        clause = text.lower().rfind(" and ")
        if clause > 20:  # end on a complete phrase: "Fix posting and add the customer's" -> "Fix posting"
            text = text[:clause]
        text = re.sub(r"(\s+(and|or|the|a|an|to|of|for|in|on|with|&|-))+$", "", text, flags=re.IGNORECASE)
    return text.rstrip(" .,-") or "Unnamed"


def ticket_folder(customer: str, item_id, description: str, doc_type: str | None = None) -> list:
    """['Customer', 'ID - Description', 'Document type'] below the root folder."""
    description = re.sub(rf"^{re.escape(customer)}\s*-\s*", "", description.strip(), flags=re.IGNORECASE)
    parts = [safe_segment(customer), safe_segment(f"{item_id} - {description}")]
    if doc_type:
        if doc_type not in DOC_TYPES:
            raise ValueError(f"Unknown document type {doc_type!r}: use one of {DOC_TYPES}")
        parts.append(doc_type)
    return parts


class SharePointClient:
    def __init__(self):
        self.site_url = os.getenv("SHAREPOINT_SITE_URL", "").rstrip("/")
        self.library = os.getenv("SHAREPOINT_LIBRARY", "Documents")
        self.root = [safe_segment(p) for p in os.getenv("SHAREPOINT_ROOT_FOLDER", "Support").split("/") if p.strip()]
        self.client_id = os.getenv("GRAPH_CLIENT_ID", "")
        self.tenant_id = os.getenv("GRAPH_TENANT_ID", "")
        self.secret = os.getenv("GRAPH_CLIENT_SECRET", "")
        self.static_token = os.getenv("GRAPH_ACCESS_TOKEN", "")
        self.live = bool(self.site_url and (self.static_token or (self.client_id and self.tenant_id)))
        self._drive_id = None

    @property
    def mode(self) -> str:
        return "live" if self.live else "dry-run"

    # ---------------- authentication (same app and token cache as the mail client) ----------------
    def _public_app(self):
        import msal

        cache = msal.SerializableTokenCache()
        if TOKEN_CACHE.exists():
            cache.deserialize(TOKEN_CACHE.read_text(encoding="utf-8"))
        app = msal.PublicClientApplication(
            self.client_id, authority=f"https://login.microsoftonline.com/{self.tenant_id}", token_cache=cache)
        return app, cache

    def sign_in(self) -> str:
        """Delegated sign-in with a device code for SharePoint access. Cached for later runs."""
        if not self.live or self.static_token or self.secret:
            return f"No interactive sign-in needed ({self.mode}{', app-only' if self.secret else ''})."
        app, cache = self._public_app()
        accounts = app.get_accounts()
        if accounts and app.acquire_token_silent(SCOPES, account=accounts[0]):
            return f"SharePoint access ready for {accounts[0]['username']}"
        flow = app.initiate_device_flow(scopes=SCOPES)
        if "user_code" not in flow:
            raise RuntimeError(f"Could not start the device sign-in: {flow.get('error_description', flow)}")
        print(flow["message"], flush=True)
        result = app.acquire_token_by_device_flow(flow)
        if "access_token" not in result:
            raise RuntimeError(f"Sign-in failed: {result.get('error_description', result)}")
        if cache.has_state_changed:
            TOKEN_CACHE.write_text(cache.serialize(), encoding="utf-8")
        return f"Signed in as {result.get('id_token_claims', {}).get('preferred_username', 'unknown user')}"

    def _token(self) -> str:
        if self.static_token:
            return self.static_token
        import msal

        if self.secret:
            app = msal.ConfidentialClientApplication(
                self.client_id, client_credential=self.secret,
                authority=f"https://login.microsoftonline.com/{self.tenant_id}")
            result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        else:
            app, cache = self._public_app()
            accounts = app.get_accounts()
            result = app.acquire_token_silent(SCOPES, account=accounts[0]) if accounts else None
            if not result:
                raise PermissionError("No SharePoint access yet: run SharePointClient().sign_in() in the notebook.")
            if cache.has_state_changed:
                TOKEN_CACHE.write_text(cache.serialize(), encoding="utf-8")
        if "access_token" not in result:
            raise PermissionError(f"Could not get a Graph token: {result.get('error_description', result)}")
        return result["access_token"]

    def _graph(self, method: str, path: str, ok=(), **kwargs) -> requests.Response:
        response = requests.request(method, f"{GRAPH_URL}{path}", timeout=120,
                                    headers={"Authorization": f"Bearer {self._token()}", **kwargs.pop("headers", {})},
                                    **kwargs)
        if not response.ok and response.status_code not in ok:
            raise requests.HTTPError(f"Graph {method} {path} -> {response.status_code}: {response.text[:400]}",
                                     response=response)
        return response

    # ---------------- site, library and folders ----------------
    def drive_id(self) -> str:
        """The document library of the site (matched by name, e.g. 'Documents' = 'Shared Documents')."""
        if self._drive_id:
            return self._drive_id
        url = urlparse(self.site_url)
        site = self._graph("GET", f"/sites/{url.hostname}:{quote(url.path)}").json()
        drives = self._graph("GET", f"/sites/{site['id']}/drives").json()["value"]
        wanted = self.library.lower()
        for drive in drives:
            names = {drive.get("name", "").lower(), drive.get("webUrl", "").rstrip("/").rsplit("/", 1)[-1].replace("%20", " ").lower()}
            if wanted in names or (wanted == "documents" and "shared documents" in names):
                self._drive_id = drive["id"]
                return self._drive_id
        raise ValueError(f"No library named {self.library!r} on {self.site_url}: {[d.get('name') for d in drives]}")

    @staticmethod
    def _path(parts: list) -> str:
        return "/".join(quote(part, safe="") for part in parts)

    def ensure_folder(self, parts: list) -> dict:
        """Create the folders below the root that do not exist yet; return the last folder."""
        full = self.root + [safe_segment(p, 120) for p in parts]
        if not self.live:
            folder = DRY_RUN_DIR.joinpath(*full)
            folder.mkdir(parents=True, exist_ok=True)
            return {"path": "/".join(full), "web_url": folder.resolve().as_uri()}

        drive = self.drive_id()
        item = None
        for depth in range(1, len(full) + 1):
            found = self._graph("GET", f"/drives/{drive}/root:/{self._path(full[:depth])}", ok=(404,))
            if found.status_code != 404:
                item = found.json()
                continue
            parent = f"/drives/{drive}/root" + (f":/{self._path(full[:depth - 1])}:" if depth > 1 else "")
            created = self._graph("POST", f"{parent}/children", ok=(409,), json={
                "name": full[depth - 1], "folder": {}, "@microsoft.graph.conflictBehavior": "fail"})
            item = created.json() if created.status_code != 409 else \
                self._graph("GET", f"/drives/{drive}/root:/{self._path(full[:depth])}").json()  # created meanwhile
        return {"path": "/".join(full), "web_url": item["webUrl"]}

    @staticmethod
    def _uploadable(path: str) -> Path:
        file = Path(path).resolve()
        if not any(file.is_relative_to(folder.resolve()) for folder in UPLOADABLE_DIRS):
            raise PermissionError(f"Only files from output/ or attachments/ can be uploaded: {path}")
        if not file.is_file():
            raise FileNotFoundError(f"File not found: {path}")
        if file.stat().st_size > MAX_SIMPLE_UPLOAD_BYTES:
            raise ValueError(f"{file.name} is larger than 250 MB.")
        return file

    def upload(self, parts: list, path: str) -> dict:
        """Upload one file into the folder (a new version when the file already exists)."""
        file = self._uploadable(path)
        folder = self.ensure_folder(parts)
        if not self.live:
            target = DRY_RUN_DIR.joinpath(*folder["path"].split("/"), file.name)
            shutil.copy2(file, target)
            return {"name": file.name, "size": file.stat().st_size, "web_url": target.resolve().as_uri()}
        item = self._graph(
            "PUT", f"/drives/{self.drive_id()}/root:/{self._path(folder['path'].split('/') + [file.name])}:/content",
            data=file.read_bytes(), headers={"Content-Type": "application/octet-stream"}).json()
        return {"name": file.name, "size": item.get("size"), "web_url": item["webUrl"]}

    def file_documents(self, customer: str, item_id, description: str, doc_type: str, paths: list) -> dict:
        """File documents as <root>/<Customer>/<ID - Description>/<Document type>/ and report every file."""
        parts = ticket_folder(customer, item_id, description, doc_type)  # validates the type before creating folders
        ticket = self.ensure_folder(parts[:-1])
        files = []
        for path in paths:
            try:
                files.append({**self.upload(parts, path), "status": "uploaded"})
            except Exception as error:  # report every failed file instead of hiding it
                files.append({"name": Path(path).name, "status": f"failed: {error}"})
        return {"mode": self.mode, "ticket_folder": ticket, "folder": "/".join(self.root + parts), "files": files}

In [ ]:
%%writefile sharepoint_server.py
"""SharePoint MCP server: files generated documents and customer attachments in the support library as
<root>/<Customer>/<ID - Description>/<Document type>/<file>."""
import json
import sys

from mcp.server.fastmcp import FastMCP

from sharepoint_client import DOC_TYPES, SharePointClient

mcp = FastMCP("sharepoint")
sharepoint = SharePointClient()


def log(message: str) -> None:
    print(f"[sharepoint tool:{sharepoint.mode}] {message}", file=sys.stderr, flush=True)


@mcp.tool()
def file_documents(customer: str, item_id: str, description: str, doc_type: str, file_paths: list[str]) -> str:
    """Upload documents to <root>/<customer>/<item_id - description>/<doc_type>/, creating missing folders.
    item_id: the User Story ID (support) or the request ID (delivery). description: short ticket description.
    doc_type: 'Change Request', 'Design', 'Test' or 'Customer Files'.
    file_paths: files generated in output/ or downloaded from the email in attachments/ (others are refused).
    An existing file with the same name gets a new version."""
    log(f"file_documents(customer={customer!r}, item_id={item_id!r}, doc_type={doc_type!r}, files={len(file_paths)})")
    if doc_type not in DOC_TYPES:
        raise ValueError(f"doc_type must be one of {DOC_TYPES}")
    return json.dumps(sharepoint.file_documents(customer, item_id, description, doc_type, file_paths))


@mcp.tool()
def ticket_folder_url(customer: str, item_id: str, description: str) -> str:
    """Create (if needed) and return the ticket folder <root>/<customer>/<item_id - description>/."""
    log(f"ticket_folder_url(customer={customer!r}, item_id={item_id!r})")
    from sharepoint_client import ticket_folder

    return json.dumps(sharepoint.ensure_folder(ticket_folder(customer, item_id, description)))


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
import importlib

import sharepoint_client

importlib.reload(sharepoint_client)
sharepoint = sharepoint_client.SharePointClient()
print("SharePoint mode:", sharepoint.mode)
print(sharepoint.sign_in())  # live + delegated: device-code sign-in the first time, cached afterwards
MCP_SERVERS["sharepoint"] = mcp_server("sharepoint_server.py")

In [ ]:
sharepoint_folder = None
if support_ticket:
    story_id = support_ticket["story"]["id"]
    folder_args = {"customer": customer, "item_id": str(story_id), "description": ticket["title"]}
    calls = []
    if attachment_files:
        calls.append(("file_documents", {**folder_args, "doc_type": "Customer Files",
                                         "file_paths": [f["path"] for f in attachment_files]}))
    if change_request:
        calls.append(("file_documents", {**folder_args, "doc_type": "Change Request", "file_paths": [change_request["path"]]}))
    calls.append(("ticket_folder_url", folder_args))  # the ticket folder exists even before any document is filed
    *filings, sharepoint_folder = call_mcp_tools("sharepoint", calls)

    for filing in filings:
        for file in filing.get("files", []):
            print(f"{filing['folder']}/{file['name']}: {file['status']}")
    link = call_mcp_tool("devops", "add_document_link", work_item_id=story_id, url=sharepoint_folder["web_url"],
                         approval_id=ticket_approval_id, message_id=support_email["message_id"])
    print(f"\nTicket folder: {sharepoint_folder['path']}\nLinked on #{story_id}: {link}")
else:
    print("No ticket was created, so nothing is filed.")

### Acknowledgement Email

The **Support Reply Agent** writes the two free-text parts (case description and next action plan); the rest
is the fixed template of the EmailToWorkItem app, with the ticket number, the scope agreement and the next
contact date (next business day). The subject is `Customer - task | ticket number`. The email is saved as a
reply-all draft after the first approval and sent only after a second approval of that exact draft.

In [ ]:
class SupportReply(BaseModel):
    greeting_name: str = Field(description="The customer's first name")
    case_description: str = Field(description="1-2 polite sentences summarizing the issue; acknowledge attached files")
    next_action_plan: str = Field(description="1-2 sentences on how the team will investigate and resolve it")


SUPPORT_REPLY_SYSTEM = """
You write the acknowledgement email for a new Dynamics 365 F&O support ticket.
- Short, polite business language without technical jargon.
- Do not mention internal names, estimates or task IDs.
- Never promise a fix date or a solution: the next contact date is given separately.
"""


@traceable(name="Support Reply Agent")
def draft_support_reply(feedback: str = "") -> SupportReply:
    user_prompt = (
        f"Customer email from {support_email['from_name']}:\n{support_email['body']}\n\n"
        f"Attachments received: {', '.join(a['name'] for a in support_email['attachments']) or 'none'}\n\n"
        f"Ticket: {ticket['title']}\n{ticket['description']}\n\n"
        "Planned work:\n" + "\n".join(f"- {t['title']}" for t in support_tasks)
        + (f"\n\nWe are sending the customer a Change Request (written by us, in its own paragraph) for: "
           f"{'; '.join(cr_assessment.change_items)}. Do not describe it as something the customer sent."
           if change_request else "")
    )
    if feedback:
        user_prompt += f"\n\nReviewer feedback on the previous draft: {feedback}"
    return ask_structured(SupportReply, SUPPORT_REPLY_SYSTEM, user_prompt)


def render_support_reply(reply: SupportReply, ticket_id, next_contact: str, cr: dict | None = None) -> str:
    return (
        f"Dear {reply.greeting_name},\n\n"
        f"This is to inform you that a support ticket **#{ticket_id}** has been created and dispatched. "
        "Please find the details below:\n\n"
        f"**Case Description:**\n{reply.case_description}\n\n"
        f"**Next Action Plan:**\n{reply.next_action_plan}\n\n"
        + cr_reply_paragraph(cr) +
        "**Scope Agreement:**\nThis case will be considered resolved and archived once the issue is fully "
        "resolved and required access is restored (if applicable).\n\n"
        "Should any new or additional issues arise during the resolution of this incident, a separate support "
        "ticket must be created.\n\n"
        f"**Next Contact Date:**\nOn or before {next_contact}.\n\n"
        f"Thank you for your attention and cooperation.\nBest regards,\n{APPROVER}"
    )


if support_ticket:
    ticket_id = support_ticket["story"]["id"]
    next_contact = add_business_days(date.today(), 1).strftime("%B %d, %Y")
    reply = draft_support_reply()
    reply_subject = f"{ticket['title']} | {ticket_id}"
    reply_body = render_support_reply(reply, ticket_id, next_contact, change_request)
    print(f"Subject: {reply_subject}\n\n{reply_body}")
else:
    print("No ticket was created, so there is nothing to acknowledge.")

In [ ]:
if not (EMAIL_SERVER_READY and support_ticket):
    print("Skipped: sending needs the email server from notebook 03 and a created ticket.")
else:
    for attempt in range(3):
        review = human_decision(f"Acknowledgement email (draft {attempt + 1})", f"Subject: {reply_subject}\n\n{reply_body}")
        record = approval_ledger.record_decision(
            "customer_email", support_email["message_id"], APPROVER,
            {"approve": "approved", "revise": "revise", "reject": "rejected"}[review["decision"]], review["comments"],
        )
        if review["decision"] != "revise":
            break
        reply = draft_support_reply(feedback=review["comments"])
        reply_body = render_support_reply(reply, ticket_id, next_contact, change_request)

    if record["decision"] == "approved":
        draft = call_mcp_tool("email", "save_reply_draft", message_id=support_email["message_id"],
                              subject=reply_subject, body=reply_body, approval_id=record["approval_id"],
                              attachment_paths=[change_request["path"]] if change_request else [])
        if "error" in draft:
            raise RuntimeError(draft["error"])
        show("SAVED DRAFT", {k: draft[k] for k in ("draft_id", "mode", "to", "cc", "subject", "attachments", "status")})

        send_review = human_decision(f"Send this draft to {', '.join(draft['to'] + draft['cc'])}?",
                                     f"Subject: {draft['subject']}\nAttachments: {', '.join(draft['attachments']) or 'none'}",
                                     options=("approve", "reject"),
                                     live_action=draft["mode"] == "live")
        send_record = approval_ledger.record_decision(
            "send_email", draft["draft_id"], APPROVER,
            "approved" if send_review["decision"] == "approve" else "rejected", send_review["comments"],
        )
        if send_record["decision"] == "approved":
            sent = call_mcp_tool("email", "send_draft", draft_id=draft["draft_id"], approval_id=send_record["approval_id"])
            show("SENT", {k: sent[k] for k in ("to", "cc", "subject", "status", "sent_at")})
        else:
            print("Not sent: the draft is waiting in the mailbox's Drafts folder.")
    else:
        print("The acknowledgement was rejected. Nothing was saved.")

## Using a Real Azure DevOps Organization and Outlook

1. Create a Personal Access Token with the **Work Items (Read & write)** and **Project and Team (Read)** scopes.
2. Put `ADO_ORG_URL` and `ADO_PAT` in `.env`, and `ADO_PROJECT` for the delivery plan. Support tickets go to
   the project of the customer's Feature. Use `ADO_STORY_TYPE = "Product Backlog Item"` for a Scrum project.
3. Support tasks set Activity, Original Estimate, Remaining Work and the due date field `ADO_DUE_DATE_FIELD`
   (default `Custom.DueDate1`; leave it empty to skip). The User Story gets Activity, Story Points (= total
   hours), Original Estimate and Priority. Adjust `create_support_items` if your process uses other fields.
4. Connect Outlook in notebook 03 (Microsoft Graph) so the email tools read and send real emails.
5. Set `HUMAN_MODE = "interactive"`: auto mode never approves live actions (real work items or real emails).
6. Re-run the cells from **Azure DevOps Client** onwards. The MCP servers receive the settings through their
   environment.

In [ ]:
# Alternative to .env: set the values for this session only
# import getpass
# os.environ["ADO_ORG_URL"] = "https://dev.azure.com/<your-organization>"
# os.environ["ADO_PROJECT"] = "<your-project>"
# os.environ["ADO_PAT"] = getpass.getpass("Azure DevOps PAT: ")

## Observation

The Planning Agent turned six approved requirements into user stories with Given/When/Then acceptance criteria
and FUNC/TECH tasks with estimates. The coverage check confirmed that every requirement ID has at least one
acceptance criterion before anything went further.

The owner recommendation used the four signals from the planning document: role, specialization, workload and
deadline. For example, the Finance developer with the heaviest backlog was not chosen automatically; the
recommender preferred the specialised person with the most free capacity and spread the hours across the team.
The result stays a **recommendation** in the description; nobody is assigned.

The MCP server refused to create anything without a ledger approval. After approval it created the stories and
their linked child tasks in the dry-run board. With a PAT configured, the same code calls the Azure DevOps REST API.

The support flow reused the same pieces for day-to-day work. The sender's domain led to the customer's project
and support Feature, and the chosen Feature was remembered, so the next email from that customer skips the
search. The Support Planning Agent split the email into small tasks with an Activity and an estimate; due dates
followed from the estimates and owners were recommended, not assigned. After approval, the User Story was
created under the Feature with its tasks and the email's attachments. The acknowledgement email carried the
real ticket number and was sent only after a second approval of that exact draft.